# HR Pro — v6
**Anytime Home Run prediction model.**

| | |
|---|---|
| Version | v6 |
| Target | Anytime HR (player scores HR in game) |
| Unit | Player-game |
| Model | XGBoost — tuned via Bayesian search |
| Data | Statcast 2021–present + weather + game context |
| AUC | v5 baseline: 0.6348 (walk-forward: 0.6293) — retrain required |

### Changes from v5
- **Batter contact quality** — `batter_sweetspot_rate_L20`, `batter_hr_zone_rate_L20` (EV≥95 + LA 20-35°), `batter_max_ev_L20`, `batter_la_std_L20`, `batter_hr_per_fb_L20`
- **Batter swing decisions** — `batter_whiff_pct_L20`, `batter_chase_pct_L20`, `batter_zone_contact_pct_L20` from pitch description + plate location
- **Pitcher arsenal** — `pitcher_hr_per_fb_L50`, `pitcher_fb_pct_L50`, `pitcher_brk_pct_L50`, `pitcher_off_pct_L50`, `pitcher_high_fb_pct_L50`
- **Bullpen exposure proxy** — `starter_avg_ip_L5` from outs_when_up
- **Park factors** — handedness-split HR park factors via `build_savant_hr_park_factors()` + `apply_handedness_park_factor()`
- **Weather fix** — `is_dome` flag; `temperature_f` filled with 70 (not 0) for dome/retractable games; fixes `air_density` corruption
- **Dashboard** — edge/bet size rounded to whole numbers; pitcher column replaced with batter team
- **Section 14b** — dedicated SHAP analysis with prune candidates

---
### Section Map
| # | Section | Run when |
|---|---|---|
| 0 | Config & Imports | Always first |
| 1 | Statcast Pull | Initial setup + nightly |
| 2 | Weather Pull | Initial setup + nightly |
| 2b | Park Factor Pull | Once per offseason |
| 3 | Game Features | Initial setup + daily |
| 4 | Odds Pull + History | Daily |
| 5 | Lineup & Pitcher Pull | Daily |
| 6 | Player-Game Aggregation | After Statcast pull |
| 7 | Batter Rolling + Platoon Features | After Section 6 |
| 8 | Pitcher Rolling Features | After Section 6 |
| 9 | Feature Join | After Sections 7 + 8 |
| 10 | Model Train (OOS + Walk-forward) | Periodic re-evaluation |
| 10b | Full Retrain (production) | Before live season |
| 11 | Calibrate | After OOS train |
| 12 | Bet Tracker | Log + settle bets |
| 13 | Daily Command Center | Every game day |
| 14 | Mathematical Rigor Assessment | After retrain |
| 14b | SHAP Analysis | After retrain, before production |

## Section 0: Config & Imports
Run this cell first, every session.

In [1]:
# !pip install pandas numpy scikit-learn xgboost requests tqdm selenium optuna shap --quiet

In [2]:
import os, json, time, random, re, warnings, sqlite3, unicodedata
from datetime import datetime, date, timedelta
from pathlib import Path
from io import StringIO

import pandas as pd
import numpy as np
import xgboost as xgb
import requests
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.isotonic import IsotonicRegression

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('ggplot')

print(f"✅ Imports loaded — {datetime.now().strftime('%Y-%m-%d %H:%M')}")

✅ Imports loaded — 2026-05-07 02:23


In [3]:
# ── CONFIG ────────────────────────────────────────────────────────────────
BASE_DIR        = r"C:\Users\lmayn\Downloads\HR_Pro"
NRFI_LINEUP     = r"C:\Users\lmayn\Downloads\NRFI_Pro_System_Complete.tar_4\NRFI_Pro_System\lineups_master.csv"

cfg = {
    "version":      "v6",
    "base_dir":     BASE_DIR,

    "mlb_season_ranges": {
        2021: ("2021-04-01", "2021-10-03"),
        2022: ("2022-04-07", "2022-10-05"),
        2023: ("2023-03-30", "2023-10-01"),
        2024: ("2024-03-20", "2024-09-29"),
        2025: ("2025-03-18", "2025-09-28"),
        2026: ("2026-03-26", "2026-10-04"),
    },
    "season_start":       2021,
    "season_start_month": 3,
    "season_end_month":   10,

    # Directories
    "statcast_cache_dir": os.path.join(BASE_DIR, "statcast_cache_daily"),
    "weather_cache_dir":  os.path.join(BASE_DIR, "weather_cache_daily"),
    "lineup_cache_dir":   os.path.join(BASE_DIR, "lineup_cache_daily"),
    "odds_cache_dir":     os.path.join(BASE_DIR, "odds_cache_daily"),

    # Master data files
    "statcast_master":       os.path.join(BASE_DIR, "statcast_fullgame_master.csv"),
    "weather_master":        os.path.join(BASE_DIR, "weather_master.csv"),
    "park_factors":          os.path.join(BASE_DIR, "savant_hr_pf.csv"),
    "game_features_master":  os.path.join(BASE_DIR, "game_features_master.csv"),
    "player_game_master":    os.path.join(BASE_DIR, "player_game_master.csv"),
    "batter_features":       os.path.join(BASE_DIR, "batter_rolling_features.csv"),
    "platoon_features":      os.path.join(BASE_DIR, "batter_platoon_features.csv"),
    "pitcher_features":      os.path.join(BASE_DIR, "pitcher_hr_features.csv"),
    "model_features":        os.path.join(BASE_DIR, "model_features.csv"),
    "odds_master":           os.path.join(BASE_DIR, "odds_master.csv"),
    "odds_multibook_master": os.path.join(BASE_DIR, "odds_multibook_master.csv"),
    "player_id_map":         os.path.join(BASE_DIR, "player_id_map.csv"),
    "player_order_map":      os.path.join(BASE_DIR, "player_order_map.csv"),

    # Model files
    "model_dir":   os.path.join(BASE_DIR, "models"),
    "model_xgb":   os.path.join(BASE_DIR, "models", "xgb_hr_v6.json"),
    "model_meta":  os.path.join(BASE_DIR, "models", "model_meta_v6.json"),
    "calibrator":  os.path.join(BASE_DIR, "models", "isotonic_hr_v6.pkl"),

    # Bet tracking
    "bet_db": os.path.join(BASE_DIR, "data", "hr_bets_v6.db"),

    # Signal config
    "min_edge":       0.02,       # display filter only — not signal gate
    "kelly_fraction": 0.50,       # half Kelly
    "min_kelly_pct":  0.001,      # signal gate
    "max_kelly_pct":  0.03,
}

for d in ["statcast_cache_dir","weather_cache_dir","lineup_cache_dir","odds_cache_dir","model_dir"]:
    Path(cfg[d]).mkdir(parents=True, exist_ok=True)
Path(os.path.join(BASE_DIR, "data")).mkdir(parents=True, exist_ok=True)

data     = {}
features = None
model    = {}
tracker  = None

print(f"✅ Config loaded — {BASE_DIR}")

✅ Config loaded — C:\Users\lmayn\Downloads\HR_Pro


In [4]:
# ── Shared utilities ──────────────────────────────────────────────────────

def american_to_implied_prob(odds):
    if pd.isna(odds): return np.nan
    odds = float(odds)
    return 100 / (odds + 100) if odds > 0 else abs(odds) / (abs(odds) + 100)

def implied_to_american(prob):
    if pd.isna(prob) or prob <= 0 or prob >= 1: return np.nan
    return round(-prob / (1 - prob) * 100) if prob > 0.5 else round((1 - prob) / prob * 100)

def kelly_stake(edge, odds, fraction=0.25):
    if pd.isna(edge) or pd.isna(odds) or edge <= 0: return 0
    b = odds / 100 if odds > 0 else 100 / abs(odds)
    return max(0, (edge / b) * fraction)

def normalize_name(s):
    if pd.isna(s): return s
    return unicodedata.normalize("NFD", str(s)).encode("ascii", "ignore").decode().lower().strip()

def derive_barrel(launch_speed, launch_angle, launch_speed_angle=None):
    if launch_speed_angle is not None:
        lsa = pd.to_numeric(launch_speed_angle, errors="coerce")
        barrel_exact  = (lsa == 6).fillna(False).astype(int)
        ls = pd.to_numeric(launch_speed, errors="coerce")
        la = pd.to_numeric(launch_angle,  errors="coerce")
        barrel_approx = (
            ((ls >= 98) & (la >= 26) & (la <= 30)) |
            ((ls >= 99) & (la >= 25) & (la <= 31)) |
            ((ls >= 100) & (la >= 24) & (la <= 33)) |
            ((ls >= 103) & (la >= 20) & (la <= 35))
        ).fillna(False).astype(int)
        return barrel_exact.where(lsa.notna(), barrel_approx)
    ls = pd.to_numeric(launch_speed, errors="coerce")
    la = pd.to_numeric(launch_angle,  errors="coerce")
    return (
        ((ls >= 98) & (la >= 26) & (la <= 30)) |
        ((ls >= 99) & (la >= 25) & (la <= 31)) |
        ((ls >= 100) & (la >= 24) & (la <= 33)) |
        ((ls >= 103) & (la >= 20) & (la <= 35))
    ).fillna(False).astype(int)

def calibrate(p):
    if "calibrator" in model:
        return float(model["calibrator"].predict([p])[0])
    return p

def build_date_list(start_date, end_date=None):
    if end_date is None: end_date = date.today() - timedelta(days=1)
    if isinstance(start_date, str): start_date = datetime.strptime(start_date, "%Y-%m-%d").date()
    if isinstance(end_date,   str): end_date   = datetime.strptime(end_date,   "%Y-%m-%d").date()
    dates, cursor = [], start_date
    while cursor <= end_date:
        if cfg['season_start_month'] <= cursor.month <= cfg['season_end_month']:
            dates.append(cursor)
        cursor += timedelta(days=1)
    return dates

def p_two_plus_hr(p_hr_game, avg_pa=3.8):
    if pd.isna(p_hr_game) or p_hr_game <= 0: return 0.0
    p_hr_pa = 1 - (1 - p_hr_game) ** (1 / avg_pa)
    p_zero  = (1 - p_hr_pa) ** avg_pa
    p_one   = avg_pa * p_hr_pa * (1 - p_hr_pa) ** (avg_pa - 1)
    return max(0, 1 - p_zero - p_one)

def air_density_ratio(altitude_ft, temp_f=70):
    alt_m  = altitude_ft * 0.3048
    temp_k = (temp_f - 32) * 5/9 + 273.15
    t0_k   = 293.15
    M, g, R = 0.0289644, 9.80665, 8.31446
    p_ratio   = np.exp(-M * g * alt_m / (R * temp_k))
    rho_ratio = p_ratio * (t0_k / temp_k)
    return round(float(rho_ratio), 4)

print("✅ Utilities loaded")

✅ Utilities loaded


In [5]:
# ── Constants ─────────────────────────────────────────────────────────────

HR_PARK_FACTORS = {
    "ARI": 1.05, "ATL": 1.08, "BAL": 1.02, "BOS": 0.87, "CHC": 1.10,
    "CWS": 1.07, "CIN": 1.15, "CLE": 0.95, "COL": 1.25, "DET": 0.97,
    "HOU": 0.94, "KC":  1.03, "LAA": 1.01, "LAD": 1.05, "MIA": 0.88,
    "MIL": 1.04, "MIN": 1.08, "NYM": 1.06, "NYY": 1.12, "OAK": 0.91,
    "PHI": 1.10, "PIT": 0.96, "SD":  0.93, "SF":  0.88, "SEA": 0.94,
    "STL": 1.00, "TB":  0.95, "TEX": 1.08, "TOR": 1.07, "WSH": 1.03,
}

PARK_DIMENSIONS = {
    "ARI": {"LF":330,"LCF":374,"CF":407,"RCF":413,"RF":334},
    "ATL": {"LF":335,"LCF":385,"CF":400,"RCF":375,"RF":325},
    "BAL": {"LF":333,"LCF":364,"CF":400,"RCF":373,"RF":318},
    "BOS": {"LF":310,"LCF":379,"CF":390,"RCF":380,"RF":302},
    "CHC": {"LF":355,"LCF":368,"CF":400,"RCF":368,"RF":353},
    "CWS": {"LF":330,"LCF":375,"CF":400,"RCF":375,"RF":335},
    "CIN": {"LF":328,"LCF":379,"CF":404,"RCF":370,"RF":325},
    "CLE": {"LF":325,"LCF":370,"CF":400,"RCF":375,"RF":325},
    "COL": {"LF":347,"LCF":390,"CF":415,"RCF":375,"RF":350},
    "DET": {"LF":345,"LCF":370,"CF":412,"RCF":365,"RF":330},
    "HOU": {"LF":315,"LCF":362,"CF":409,"RCF":373,"RF":326},
    "KC":  {"LF":330,"LCF":387,"CF":410,"RCF":387,"RF":330},
    "LAA": {"LF":347,"LCF":390,"CF":396,"RCF":370,"RF":350},
    "LAD": {"LF":330,"LCF":385,"CF":400,"RCF":385,"RF":330},
    "MIA": {"LF":344,"LCF":386,"CF":400,"RCF":387,"RF":335},
    "MIL": {"LF":344,"LCF":371,"CF":400,"RCF":374,"RF":345},
    "MIN": {"LF":339,"LCF":377,"CF":411,"RCF":367,"RF":328},
    "NYM": {"LF":335,"LCF":370,"CF":408,"RCF":375,"RF":330},
    "NYY": {"LF":318,"LCF":399,"CF":408,"RCF":385,"RF":314},
    "OAK": {"LF":330,"LCF":388,"CF":400,"RCF":388,"RF":330},
    "PHI": {"LF":329,"LCF":374,"CF":401,"RCF":369,"RF":330},
    "PIT": {"LF":325,"LCF":383,"CF":399,"RCF":375,"RF":320},
    "SD":  {"LF":334,"LCF":390,"CF":396,"RCF":391,"RF":322},
    "SF":  {"LF":339,"LCF":364,"CF":399,"RCF":421,"RF":309},
    "SEA": {"LF":331,"LCF":378,"CF":401,"RCF":381,"RF":326},
    "STL": {"LF":336,"LCF":375,"CF":400,"RCF":375,"RF":335},
    "TB":  {"LF":315,"LCF":370,"CF":404,"RCF":370,"RF":322},
    "TEX": {"LF":329,"LCF":372,"CF":407,"RCF":374,"RF":326},
    "TOR": {"LF":328,"LCF":375,"CF":400,"RCF":375,"RF":328},
    "WSH": {"LF":336,"LCF":377,"CF":402,"RCF":370,"RF":335},
}

PARK_ALTITUDE = {
    "ARI": 1082, "ATL": 1050, "BAL":   50, "BOS":   20, "CHC":  595,
    "CWS":  595, "CIN":  490, "CLE":  650, "COL": 5280, "DET":  585,
    "HOU":   43, "KC":   750, "LAA":  160, "LAD":  515, "MIA":    6,
    "MIL":  635, "MIN":  815, "NYM":   20, "NYY":   20, "OAK":   20,
    "PHI":   20, "PIT":  730, "SD":    17, "SF":    10, "SEA":   17,
    "STL":  465, "TB":    6,  "TEX":  551, "TOR":  173, "WSH":   25,
}

TEAM_NAME_TO_ABBR = {
    "ARI":"ARI","AZ":"ARI","ATL":"ATL","BAL":"BAL","BOS":"BOS",
    "CHC":"CHC","CWS":"CWS","CIN":"CIN","CLE":"CLE","COL":"COL",
    "DET":"DET","HOU":"HOU","KC":"KC","LAA":"LAA","LAD":"LAD",
    "MIA":"MIA","MIL":"MIL","MIN":"MIN","NYM":"NYM","NYY":"NYY",
    "OAK":"OAK","PHI":"PHI","PIT":"PIT","SD":"SD","SF":"SF",
    "SEA":"SEA","STL":"STL","TB":"TB","TEX":"TEX","TOR":"TOR",
    "WSH":"WSH","WAS":"WSH","ATH":"OAK",
    "Arizona Diamondbacks":"ARI","Atlanta Braves":"ATL",
    "Baltimore Orioles":"BAL","Boston Red Sox":"BOS",
    "Chicago Cubs":"CHC","Chicago White Sox":"CWS",
    "Cincinnati Reds":"CIN","Cleveland Guardians":"CLE",
    "Cleveland Indians":"CLE","Colorado Rockies":"COL",
    "Detroit Tigers":"DET","Houston Astros":"HOU",
    "Kansas City Royals":"KC","Los Angeles Angels":"LAA",
    "Los Angeles Dodgers":"LAD","Miami Marlins":"MIA",
    "Milwaukee Brewers":"MIL","Minnesota Twins":"MIN",
    "New York Mets":"NYM","New York Yankees":"NYY",
    "Oakland Athletics":"OAK","Athletics":"OAK",
    "Philadelphia Phillies":"PHI","Pittsburgh Pirates":"PIT",
    "San Diego Padres":"SD","San Francisco Giants":"SF",
    "Seattle Mariners":"SEA","St. Louis Cardinals":"STL",
    "Tampa Bay Rays":"TB","Texas Rangers":"TEX",
    "Toronto Blue Jays":"TOR","Washington Nationals":"WSH",
}

STADIUMS = {
    "ARI": (33.4453,-112.0667,"retractable",-7), "ATL": (33.8908,-84.4678,"open",-4),
    "BAL": (39.2838,-76.6218,"open",-4),          "BOS": (42.3467,-71.0972,"open",-4),
    "CHC": (41.9484,-87.6553,"open",-5),          "CWS": (41.8299,-87.6338,"open",-5),
    "CIN": (39.0979,-84.5082,"open",-4),          "CLE": (41.4962,-81.6852,"open",-4),
    "COL": (39.7559,-104.9942,"open",-6),         "DET": (42.3390,-83.0485,"open",-4),
    "HOU": (29.7573,-95.3555,"retractable",-5),   "KC":  (39.0517,-94.4803,"open",-5),
    "LAA": (33.8003,-117.8827,"open",-7),         "LAD": (34.0739,-118.2400,"open",-7),
    "MIA": (25.7781,-80.2197,"retractable",-4),   "MIL": (43.0280,-87.9712,"retractable",-5),
    "MIN": (44.9817,-93.2778,"open",-5),          "NYM": (40.7571,-73.8458,"open",-4),
    "NYY": (40.8296,-73.9262,"open",-4),          "OAK": (37.7516,-122.2005,"open",-7),
    "PHI": (39.9061,-75.1665,"open",-4),          "PIT": (40.4469,-80.0057,"open",-4),
    "SD":  (32.7076,-117.1570,"open",-7),         "SF":  (37.7786,-122.3893,"open",-7),
    "SEA": (47.5914,-122.3325,"retractable",-7),  "STL": (38.6226,-90.1928,"open",-5),
    "TB":  (27.7683,-82.6534,"dome",-4),          "TEX": (32.7512,-97.0832,"retractable",-5),
    "TOR": (43.6414,-79.3894,"retractable",-4),   "WSH": (38.8730,-77.0074,"open",-4),
}

WIND_OUT_PARKS = {"CHC","COL","TEX","LAD"}
WIND_IN_PARKS  = {"CHC","COL","SF","BOS"}
EXPECTED_PA    = {1:4.3,2:4.2,3:4.1,4:4.0,5:3.9,6:3.8,7:3.7,8:3.6,9:3.5}

LEAGUE_HR_BY_MATCHUP = {
    ("L","L"): 0.0238, ("L","R"): 0.0322,
    ("R","L"): 0.0325, ("R","R"): 0.0303,
}

def get_pull_side_dist(home_abbr, stand):
    dims = PARK_DIMENSIONS.get(home_abbr)
    if not dims: return np.nan
    return dims["LF"] if stand == "R" else dims["RF"]

def order_to_expected_pa(pos):
    if pd.isna(pos): return 3.8
    lower = max(1, min(9, int(pos)))
    upper = min(9, lower + 1)
    frac  = pos - int(pos)
    return round(EXPECTED_PA[lower] * (1-frac) + EXPECTED_PA[upper] * frac, 3)

# ── v6 Feature set ────────────────────────────────────────────────────────
HR_FEATURES = [
    # Batter contact (L50)
    "batter_barrel_rate_L50",
    "batter_hr_rate_L50",
    "batter_hr_rate_season",
    "batter_max_ev_L20",
    "batter_sweetspot_rate_L20",
    "batter_xwoba_season",
    # Platoon
    "batter_hr_rate_vs_hand",
    "batter_hr_rate_vs_hand_season",
    # Swing decisions
    "batter_zone_contact_pct_L20",
    "batter_chase_pct_L20",
    "batter_hr_per_fb_L20",
    # Pitcher
    "pitcher_fb_rate_L50",
    "pitcher_hard_hit_L50",
    "pitcher_xwoba_L50",
    "pitcher_barrel_rate_L50",
    "pitcher_fb_pct_L50",
    # Park / environment
    "hr_park_factor",       # drop hr_park_factor_hand — perfect duplicate (corr=1.0)
    "altitude_ft",
    "air_density",
    # Context
    "ewma_batting_order",
    "temperature_f",
    "implied_win_pct",
]

seen = set()
HR_FEATURES = [f for f in HR_FEATURES if not (f in seen or seen.add(f))]

# ── Tuned XGBoost params (Bayesian search, 200 trials, rolling2 CV) ──────
XGB_PARAMS = {
    "objective":        "binary:logistic",
    "eval_metric":      ["logloss","auc"],
    "max_depth":        3,
    "learning_rate":    0.031,
    "subsample":        0.74,
    "colsample_bytree": 0.98,
    "min_child_weight": 11,
    "reg_alpha":        1.09,
    "reg_lambda":       6.78,
    "gamma":            0.39,
    "seed":             42,
    "nthread":          12,
}

# ── API URLs ──────────────────────────────────────────────────────────────
HR_ODDS_BASE              = "https://djstrauss08.github.io/HomeRunOdds/api/v1"
DK_HR_URL                 = "https://sportsbook.draftkings.com/leagues/baseball/mlb?category=batter-props&subcategory=home-runs"
DK_GAMELINES_URL          = "https://sportsbook.draftkings.com/leagues/baseball/mlb?category=game-lines"
MLB_SCHEDULE_URL          = "https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date}&gameType=R&fields=dates,games,gamePk,gameDate,teams,away,home,team,id,name"
MLB_SCHEDULE_PITCHERS_URL = "https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date}&gameType=R&hydrate=probablePitcher&fields=dates,games,gamePk,teams,away,home,team,name,probablePitcher,id,fullName"
MLB_BOXSCORE_URL          = "https://statsapi.mlb.com/api/v1/game/{game_pk}/boxscore"
MLB_PLAYER_SEARCH         = "https://statsapi.mlb.com/api/v1/people/search?names={name}&sportId=1"
WEATHER_VARS              = "temperature_2m,windspeed_10m,winddirection_10m,precipitation,weathercode"

_session = requests.Session()
_session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})

DK_NAME_TO_ABBR = {
    "ARI":"ARI","ATL":"ATL","BAL":"BAL","BOS":"BOS","CHC":"CHC","CWS":"CWS",
    "CIN":"CIN","CLE":"CLE","COL":"COL","DET":"DET","HOU":"HOU","KC":"KC",
    "LAA":"LAA","LAD":"LAD","MIA":"MIA","MIL":"MIL","MIN":"MIN","NYM":"NYM",
    "NYY":"NYY","OAK":"OAK","PHI":"PHI","PIT":"PIT","SD":"SD","SF":"SF",
    "SEA":"SEA","STL":"STL","TB":"TB","TEX":"TEX","TOR":"TOR","WSH":"WSH",
    "A's":"OAK","AZ":"ARI","ATH":"OAK",
    "Athletics":"OAK","Diamondbacks":"ARI","Braves":"ATL","Orioles":"BAL",
    "Red Sox":"BOS","Cubs":"CHC","White Sox":"CWS","Reds":"CIN","Guardians":"CLE",
    "Rockies":"COL","Tigers":"DET","Astros":"HOU","Royals":"KC","Angels":"LAA",
    "Dodgers":"LAD","Marlins":"MIA","Brewers":"MIL","Twins":"MIN","Mets":"NYM",
    "Yankees":"NYY","Phillies":"PHI","Pirates":"PIT","Padres":"SD","Giants":"SF",
    "Mariners":"SEA","Cardinals":"STL","Rays":"TB","Rangers":"TEX",
    "Blue Jays":"TOR","Nationals":"WSH",
}

DK_NAME_OVERRIDES = {
    "jazz chisholm":    "jazz chisholm jr.",
    "michael harris":   "michael harris ii",
    "c.j. kayfus":      "cj kayfus",
    "t.j. rumfield":    "tj rumfield",
}

def clean_dk_name(name):
    m = re.match(r'^(.+?)\s*\(([A-Z]{2,3})\)\s*$', str(name))
    if m:
        player, team = m.group(1).strip(), m.group(2).strip()
        if normalize_name(player) in {"max muncy", "will smith"}:
            return f"{player} {team}"
        return player
    return name

def resolve_dk_name(raw_name):
    cleaned = normalize_name(clean_dk_name(raw_name))
    return DK_NAME_OVERRIDES.get(cleaned, cleaned)

def _dk_resolve_name(name):
    if name in DK_NAME_TO_ABBR: return DK_NAME_TO_ABBR[name]
    for fragment, abbr in DK_NAME_TO_ABBR.items():
        if fragment and fragment in name: return abbr
    return name

def _dk_american_to_int(s):
    if not s: return None
    s = str(s).replace("\u2212","-").replace("\u2013","-").strip()
    try: return int(float(s))
    except: return None

print("✅ Constants and helpers loaded")
print(f"   HR_FEATURES: {len(HR_FEATURES)} | Parks: {len(PARK_DIMENSIONS)} | Altitudes: {len(PARK_ALTITUDE)}")

✅ Constants and helpers loaded
   HR_FEATURES: 22 | Parks: 30 | Altitudes: 30


## Section 1: Statcast Pull
Full-game pitch data.

**First time:** `statcast_historical(cfg)` — 3–5 hours  
**Re-pull seasons:** `statcast_repull_recent(cfg, seasons=(2021,))`  
**Daily:** `statcast_nightly(cfg)`  
**Retry failed:** `statcast_retry_failed(cfg)`

In [6]:
HR_STATCAST_FIELDS = [
    "pitcher","batter","player_name","game_pk","game_date",
    "events","description",
    "launch_speed","launch_angle","launch_speed_angle",
    "estimated_woba_using_speedangle","woba_value","hit_distance_sc",
    "bb_type","hc_x","hc_y",
    "pitch_type","release_spin_rate","pfx_x","pfx_z",
    "plate_x","plate_z","sz_top","sz_bot",
    "home_team","away_team","p_throws","stand",
    "inning","inning_topbot","at_bat_number","outs_when_up",
]

STATCAST_URL = (
    "https://baseballsavant.mlb.com/statcast_search/csv"
    "?all=true&hfPT=&hfAB=&hfBBT=&hfPR=&hfZ=&stadium=&hfBBL=&hfNewZones="
    "&hfGT=R%7C&hfSea=&hfSit=&player_type=pitcher"
    "&hfOuts=&opponent=&pitcher_throws=&batter_stands=&hfSA="
    "&game_date_gt={date}&game_date_lt={date}"
    "&team=&position=&hfRO=&home_road=&hfFlag=&metric_1="
    "&min_pitches=0&min_results=0&group_by=name&sort_col=pitches&sort_order=desc&min_abs=0&type=details"
)

def _fetch_statcast_day(date_str, fields=None):
    if fields is None: fields = HR_STATCAST_FIELDS
    url = STATCAST_URL.format(date=date_str)
    for attempt in range(4):
        try:
            r = _session.get(url, timeout=60)
            r.raise_for_status()
            if r.text.strip().startswith("<!"): raise ValueError("HTML — rate limited")
            if len(r.text.strip()) < 50: return None
            df = pd.read_csv(StringIO(r.text), low_memory=False)
            if df.empty or "game_pk" not in df.columns: return None
            keep = [f for f in fields if f in df.columns]
            return df[keep].copy()
        except ValueError:
            time.sleep((2**attempt)*3 + random.uniform(1,3))
        except Exception as e:
            if attempt == 3: print(f"    ⚠️  {date_str}: {e}")
            time.sleep((2**attempt) + random.uniform(0.5,1.5))
    return None

def statcast_rebuild_master(cfg):
    cache_dir = Path(cfg["statcast_cache_dir"])
    dfs, empty = [], 0
    for f in sorted(cache_dir.glob("*.csv")):
        if f.stat().st_size < 10: empty += 1; continue
        try:
            df = pd.read_csv(f, low_memory=False)
            if not df.empty: dfs.append(df)
            else: empty += 1
        except: empty += 1
    if not dfs: print("⚠️  No data files"); return pd.DataFrame()
    master = pd.concat(dfs, ignore_index=True)
    master.drop_duplicates(
        subset=[c for c in ["game_pk","batter","at_bat_number"] if c in master.columns],
        inplace=True
    )
    master.to_csv(cfg["statcast_master"], index=False)
    data["statcast"] = master
    print(f"  Files: {len(dfs)} with data | {empty} empty")
    print(f"  ✅ {len(master):,} rows | {master['game_date'].min()} → {master['game_date'].max()}")
    return master

def statcast_nightly(cfg):
    yesterday = (date.today()-timedelta(days=1)).strftime("%Y-%m-%d")
    path      = Path(cfg["statcast_cache_dir"]) / f"{yesterday}.csv"
    if path.exists(): path.unlink()
    print(f"Statcast nightly: {yesterday}")
    df = _fetch_statcast_day(yesterday)
    if df is not None and not df.empty:
        df.to_csv(path, index=False)
        print(f"  ✅ {len(df):,} pitches cached")
    else:
        path.write_text("")
        print("  No data for yesterday")
    return statcast_rebuild_master(cfg)

def statcast_historical(cfg):
    cache_dir = Path(cfg["statcast_cache_dir"])
    all_dates = build_date_list(datetime(cfg["season_start"],3,20).date())
    cached    = {f.stem for f in cache_dir.glob("*.csv")}
    todo      = [d for d in all_dates if d.strftime("%Y-%m-%d") not in cached]
    print(f"Pulling {len(todo)} days ({len(all_dates)-len(todo)} cached)")
    print("⚠️  ~3-5 hours. Run overnight.")
    it = tqdm(todo) if HAS_TQDM else todo
    for d in it:
        date_str = d.strftime("%Y-%m-%d")
        path     = cache_dir / f"{date_str}.csv"
        df       = _fetch_statcast_day(date_str)
        if df is not None and not df.empty: df.to_csv(path, index=False)
        else: path.write_text("")
        time.sleep(random.uniform(2.5,3.5))
    return statcast_rebuild_master(cfg)

def statcast_repull_recent(cfg, seasons=(2024,2025)):
    cache_dir = Path(cfg["statcast_cache_dir"])
    dates = []
    for season in seasons:
        if season not in cfg["mlb_season_ranges"]: continue
        start, end = cfg["mlb_season_ranges"][season]
        d     = datetime.strptime(start, "%Y-%m-%d").date()
        end_d = min(datetime.strptime(end, "%Y-%m-%d").date(), date.today()-timedelta(days=1))
        while d <= end_d:
            dates.append(d)
            d += timedelta(days=1)
    deleted = 0
    for d in dates:
        p = cache_dir / f"{d.strftime('%Y-%m-%d')}.csv"
        if p.exists(): p.unlink(); deleted += 1
    print(f"Re-pulling {len(dates)} days for {seasons} | Deleted {deleted} cache files")
    print("⚠️  ~1 hour per season. Run overnight.")
    it = tqdm(dates) if HAS_TQDM else dates
    for d in it:
        date_str = d.strftime("%Y-%m-%d")
        path     = cache_dir / f"{date_str}.csv"
        df       = _fetch_statcast_day(date_str)
        if df is not None and not df.empty: df.to_csv(path, index=False)
        else: path.write_text("")
        time.sleep(random.uniform(2.5,3.5))
    return statcast_rebuild_master(cfg)

def statcast_retry_failed(cfg):
    cache_dir = Path(cfg["statcast_cache_dir"])
    failed    = [f.stem for f in cache_dir.glob("*.csv") if f.stat().st_size == 0]
    print(f"Retrying {len(failed)} failed days...")
    for date_str in sorted(failed):
        df   = _fetch_statcast_day(date_str)
        path = cache_dir / f"{date_str}.csv"
        if df is not None and not df.empty:
            df.to_csv(path, index=False)
            print(f"  ✅ {date_str}: {len(df):,} pitches")
        else:
            print(f"  ⚠️  {date_str}: failed again")
        time.sleep(15)
    return statcast_rebuild_master(cfg)

# Load existing
if Path(cfg["statcast_master"]).exists():
    data["statcast"] = pd.read_csv(cfg["statcast_master"], low_memory=False)
    sc = data["statcast"]
    new_fields = [c for c in ["bb_type","hc_x","pitch_type","outs_when_up"] if c in sc.columns]
    print(f"✅ Statcast loaded: {len(sc):,} rows | {sc['game_date'].min()} → {sc['game_date'].max()}")
    print(f"   Fields present: {new_fields}")
else:
    print("ℹ️  No Statcast master — run statcast_historical(cfg)")

✅ Statcast loaded: 926,980 rows | 2021-04-01 → 2026-04-14
   Fields present: ['bb_type', 'hc_x', 'pitch_type', 'outs_when_up']


## Section 2: Weather Pull

**First time:** `weather_historical(cfg)`  
**Daily:** `weather_nightly(cfg)`

In [7]:
def _get_games_for_date(date_str):
    try:
        r = _session.get(MLB_SCHEDULE_URL.format(date=date_str), timeout=30)
        r.raise_for_status()
    except: return []
    games = []
    for d in r.json().get("dates",[]):
        for g in d.get("games",[]):
            games.append({
                "game_pk":        g["gamePk"],
                "game_date":      date_str,
                "game_time_utc":  g.get("gameDate",""),
                "away_team_name": g["teams"]["away"]["team"]["name"],
                "home_team_name": g["teams"]["home"]["team"]["name"],
            })
    return games

def _fetch_weather(lat, lon, date_str, hour_utc, is_forecast=False):
    if is_forecast:
        base   = "https://api.open-meteo.com/v1/forecast"
        params = {"latitude":lat,"longitude":lon,"hourly":WEATHER_VARS,"timezone":"UTC","forecast_days":3}
    else:
        base   = "https://archive-api.open-meteo.com/v1/archive"
        params = {"latitude":lat,"longitude":lon,"hourly":WEATHER_VARS,"start_date":date_str,"end_date":date_str,"timezone":"UTC"}
    for attempt in range(4):
        try:
            r      = _session.get(base, params=params, timeout=30)
            r.raise_for_status()
            hourly = r.json().get("hourly",{})
            times  = hourly.get("time",[])
            target = f"{date_str}T{hour_utc:02d}:00"
            idx    = times.index(target) if target in times else (
                min(range(len(times)), key=lambda i: abs(int(times[i][11:13])-hour_utc)) if times else 0
            )
            dirs = ["N","NE","E","SE","S","SW","W","NW"]
            deg  = hourly["winddirection_10m"][idx]
            return {
                "temperature_f":    round(hourly["temperature_2m"][idx]*9/5+32, 1),
                "wind_speed_mph":   round(hourly["windspeed_10m"][idx]*0.621371, 1),
                "wind_dir_degrees": deg,
                "wind_direction":   dirs[round(deg/45)%8] if deg is not None else None,
                "precipitation_in": round(hourly["precipitation"][idx]*0.0393701, 3),
                "weather_code":     hourly.get("weathercode",[None])[idx],
            }
        except: time.sleep((2**attempt)+random.uniform(0.5,1.0))
    return None

def _pull_weather_date(date_str, is_forecast=False):
    games = _get_games_for_date(date_str)
    if not games: return pd.DataFrame()
    rows = []
    for game in games:
        abbrev = TEAM_NAME_TO_ABBR.get(game["home_team_name"])
        if not abbrev or abbrev not in STADIUMS: continue
        lat, lon, roof, tz = STADIUMS[abbrev]
        try:
            game_dt  = datetime.strptime(game["game_time_utc"], "%Y-%m-%dT%H:%M:%SZ")
            hour_utc = game_dt.hour
        except: hour_utc = 23
        row = {"game_pk":game["game_pk"],"game_date":date_str,
               "home_team":abbrev,"away_team":TEAM_NAME_TO_ABBR.get(game["away_team_name"],""),
               "game_time_utc":game.get("game_time_utc",""),"roof":roof,"is_outdoor":int(roof=="open")}
        if roof in ("dome","retractable"):
            # v6: set temperature_f=None (not 0) so fillna logic in build_features works correctly
            row.update({"temperature_f":None,"wind_speed_mph":None,"wind_dir_degrees":None,
                        "wind_direction":None,"precipitation_in":None,"weather_code":None,
                        "wind_out":0,"wind_in":0,"is_cold":0,"is_hot":0,"high_wind":0})
        else:
            wx = _fetch_weather(lat, lon, date_str, hour_utc, is_forecast)
            if wx:
                spd = wx["wind_speed_mph"]
                row.update({**wx,
                    "wind_out": int(abbrev in WIND_OUT_PARKS and spd > 10),
                    "wind_in":  int(abbrev in WIND_IN_PARKS  and spd > 10),
                    "is_cold":  int(wx["temperature_f"] < 50),
                    "is_hot":   int(wx["temperature_f"] > 85),
                    "high_wind":int(spd > 15),
                })
            else:
                row.update({k:None for k in ["temperature_f","wind_speed_mph","wind_dir_degrees",
                            "wind_direction","precipitation_in","weather_code",
                            "wind_out","wind_in","is_cold","is_hot","high_wind"]})
        rows.append(row)
        time.sleep(random.uniform(0.2,0.5))
    return pd.DataFrame(rows)

def weather_rebuild_master(cfg):
    frames = []
    for f in sorted(Path(cfg["weather_cache_dir"]).glob("*.csv")):
        try:
            df = pd.read_csv(f)
            if not df.empty: frames.append(df)
        except: pass
    if not frames: print("  ⚠️  No weather cache files"); return
    master = pd.concat(frames, ignore_index=True).drop_duplicates("game_pk")
    master.to_csv(cfg["weather_master"], index=False)
    data["weather"] = master
    print(f"  ✅ Weather: {len(master):,} games")

def weather_historical(cfg):
    cache_dir = Path(cfg["weather_cache_dir"])
    all_dates = build_date_list(datetime(cfg["season_start"],3,20).date())
    cached    = {f.stem for f in cache_dir.glob("*.csv")}
    todo      = [d for d in all_dates if d.strftime("%Y-%m-%d") not in cached]
    print(f"Weather historical: {len(todo)} days")
    it = tqdm(todo) if HAS_TQDM else todo
    for i, d in enumerate(it):
        date_str = d.strftime("%Y-%m-%d")
        path     = cache_dir / f"{date_str}.csv"
        day_df   = _pull_weather_date(date_str, is_forecast=False)
        if not day_df.empty: day_df.to_csv(path, index=False)
        else: pd.DataFrame().to_csv(path, index=False)
        if i % 50 == 0: print(f"  {date_str}")
        time.sleep(random.uniform(0.5,1.0))
    weather_rebuild_master(cfg)

def weather_nightly(cfg):
    yesterday = (date.today()-timedelta(days=1)).strftime("%Y-%m-%d")
    print(f"Weather nightly: {yesterday}")
    day_df = _pull_weather_date(yesterday, is_forecast=False)
    if not day_df.empty:
        (Path(cfg["weather_cache_dir"]) / f"{yesterday}.csv").write_text(day_df.to_csv(index=False))
        print(f"  ✅ {len(day_df)} games")
    weather_rebuild_master(cfg)

def weather_live(cfg):
    today_str = date.today().strftime("%Y-%m-%d")
    print(f"Live weather: {today_str}")
    day_df = _pull_weather_date(today_str, is_forecast=True)
    if not day_df.empty:
        outdoor = day_df[day_df["is_outdoor"]==1]
        if not outdoor.empty:
            cols = ["home_team","away_team","temperature_f","wind_speed_mph","wind_direction","wind_out","wind_in"]
            print(outdoor[[c for c in cols if c in outdoor.columns]].to_string(index=False))
    data["weather_today"] = day_df
    return day_df

# Load existing
if Path(cfg["weather_master"]).exists():
    data["weather"] = pd.read_csv(cfg["weather_master"])
    print(f"✅ Weather loaded: {len(data['weather']):,} game rows")
else:
    print("ℹ️  No weather master — run weather_historical(cfg)")
print("✅ Weather functions defined")

✅ Weather loaded: 12,334 game rows
✅ Weather functions defined


## Section 2b: Park Factor Pull (NEW v6)
Handedness-split HR park factors from Baseball Savant.

**Run once per offseason:** `build_savant_hr_park_factors()`
**Requires:** Chrome + Selenium (same driver used elsewhere)

In [8]:
# ── Handedness-split HR park factors — NEW v6 ────────────────────────────

SAVANT_PF_TEAM_MAP = {
    "Arizona Diamondbacks": "ARI", "Atlanta Braves": "ATL",
    "Baltimore Orioles": "BAL",    "Boston Red Sox": "BOS",
    "Chicago Cubs": "CHC",         "Chicago White Sox": "CWS",
    "Cincinnati Reds": "CIN",      "Cleveland Guardians": "CLE",
    "Cleveland Indians": "CLE",    "Colorado Rockies": "COL",
    "Detroit Tigers": "DET",       "Houston Astros": "HOU",
    "Kansas City Royals": "KC",    "Los Angeles Angels": "LAA",
    "Los Angeles Dodgers": "LAD",  "Miami Marlins": "MIA",
    "Milwaukee Brewers": "MIL",    "Minnesota Twins": "MIN",
    "New York Mets": "NYM",        "New York Yankees": "NYY",
    "Oakland Athletics": "OAK",    "Athletics": "OAK",
    "Philadelphia Phillies": "PHI","Pittsburgh Pirates": "PIT",
    "San Diego Padres": "SD",      "San Francisco Giants": "SF",
    "Seattle Mariners": "SEA",     "St. Louis Cardinals": "STL",
    "Tampa Bay Rays": "TB",        "Texas Rangers": "TEX",
    "Toronto Blue Jays": "TOR",    "Washington Nationals": "WSH",
}

def _init_chrome_driver():
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=opts)

def fetch_savant_hr_pf_selenium(year, driver=None):
    owns_driver = driver is None
    if owns_driver: driver = _init_chrome_driver()
    
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.common.by import By

    frames = []
    for stand, batter_hand in [("L", "L"), ("R", "R")]:
        url = (f"https://baseballsavant.mlb.com/leaderboard/statcast-park-factors"
               f"?type=year&year={year}&stat=hr&condition=same&rolling=no"
               f"&bathand={batter_hand}")
        try:
            driver.get(url)
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr"))
            )
            time.sleep(1.5)
            tables = pd.read_html(driver.page_source)
            if not tables: continue
            df = tables[0]
            df = df.dropna(how="all").reset_index(drop=True)
            df.columns = [str(c).strip() for c in df.columns]
            print(f"  DEBUG {year} {stand} columns: {df.columns.tolist()}")
            print(f"  DEBUG row 0: {df.iloc[0].tolist()}")
        except Exception as e:
            print(f"  ❌ {year}/{stand}: {e}")
    
    if owns_driver: driver.quit()
    return pd.DataFrame()

def build_savant_hr_park_factors(seasons=range(2021, 2027), output_path=None):
    """Pull and save handedness-split HR park factors. Run once per offseason."""    
    if output_path is None: output_path = cfg["park_factors"]
    driver = _init_chrome_driver()
    frames = []
    try:
        for year in seasons:
            print(f"  Fetching park factors {year}...")
            df = fetch_savant_hr_pf_selenium(year, driver=driver)
            if not df.empty: frames.append(df)
            time.sleep(1.0)
    finally:
        driver.quit()
    if not frames: print("  ❌ No data retrieved"); return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(output_path, index=False)
    data["park_factors"] = out
    print(f"✅ Saved {len(out)} rows → {output_path}")
    return out

def apply_handedness_park_factor(df, pf_df=None):
    """Join handedness-split park factors. Falls back to static hr_park_factor on miss."""    
    if pf_df is None: pf_df = data.get("park_factors", pd.DataFrame())
    if pf_df is None or pf_df.empty:
        df["hr_park_factor_hand"] = df.get("hr_park_factor", 1.0)
        return df
    pf = pf_df.copy()
    pf["season"] = pf["season"].astype(int)
    if "season" not in df.columns:
        df["season"] = pd.to_datetime(df["game_date"]).dt.year
    df["season"] = df["season"].astype(int)
    # For current season: fall back to prior year if not yet published
    max_pf_season = pf["season"].max()
    df["pf_season"] = df["season"].clip(upper=max_pf_season)
    df = df.merge(
        pf[["season","team_abbr","stand","hr_factor"]].rename(
            columns={"team_abbr":"home_abbr","hr_factor":"hr_park_factor_hand","season":"pf_season"}
        ),
        on=["pf_season","home_abbr","stand"], how="left"
    )
    df = df.drop(columns=["pf_season"], errors="ignore")
    missing = df["hr_park_factor_hand"].isna().sum()
    if missing:
        print(f"  ⚠️  {missing} rows using static hr_park_factor fallback")
        df["hr_park_factor_hand"] = df["hr_park_factor_hand"].fillna(df.get("hr_park_factor", 1.0))
    return df

# Load existing park factors
if Path(cfg["park_factors"]).exists():
    data["park_factors"] = pd.read_csv(cfg["park_factors"])
    print(f"✅ Park factors loaded: {len(data['park_factors']):,} rows")
else:
    print("ℹ️  No park factors — run build_savant_hr_park_factors()")
print("✅ Park factor functions defined")

ℹ️  No park factors — run build_savant_hr_park_factors()
✅ Park factor functions defined


## Section 3: Game Features
Native pipeline + DK moneylines.

**First time:** `build_game_features_native(cfg)` then `append_dk_moneylines(cfg)`  
**Daily:** `append_dk_moneylines(cfg)`

In [9]:
def _build_win_pct(sc):
    print("  win_pct: starting...")
    if "post_home_score" in sc.columns and "post_away_score" in sc.columns:
        games = sc.groupby(["game_pk","game_date","home_team","away_team"]).agg(
            home_score=("post_home_score","max"),
            away_score=("post_away_score","max"),
        ).reset_index()
        games["home_winner"] = (games["home_score"] > games["away_score"]).astype(int)
    else:
        hr = sc[sc["events"]=="home_run"].copy()
        home_hr = hr[hr["inning_topbot"]=="Bot"].groupby("game_pk").size().rename("home_hr")
        away_hr = hr[hr["inning_topbot"]=="Top"].groupby("game_pk").size().rename("away_hr")
        games = sc.groupby(["game_pk","game_date","home_team","away_team"]).size().reset_index()
        games = games[["game_pk","game_date","home_team","away_team"]]
        games = games.join(home_hr, on="game_pk").join(away_hr, on="game_pk")
        games["home_hr"]     = games["home_hr"].fillna(0)
        games["away_hr"]     = games["away_hr"].fillna(0)
        games["home_winner"] = (games["home_hr"] >= games["away_hr"]).astype(int)
    print(f"  win_pct: {len(games):,} games built")
    games["game_date"] = pd.to_datetime(games["game_date"])
    games["season"]    = games["game_date"].dt.year
    games = games.sort_values(["home_team","game_date"]).reset_index(drop=True)
    games["home_win_pct_to_date"] = games.groupby(["home_team","season"])["home_winner"].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    games["away_win_pct_to_date"] = games.groupby(["away_team","season"])["home_winner"].transform(
        lambda x: (1-x).shift(1).expanding().mean()
    )
    print("  win_pct: done")
    return games[["game_pk","home_win_pct_to_date","away_win_pct_to_date"]]

def build_game_features_native(cfg, statcast_df=None):
    """Build game_features_master from Statcast. Covers all years including 2026+."""
    if statcast_df is None: statcast_df = data.get("statcast")
    if statcast_df is None or statcast_df.empty: print("No Statcast"); return pd.DataFrame()

    sc = statcast_df.copy()
    sc["game_date"] = pd.to_datetime(sc["game_date"])
    pa = sc[sc["events"].notna()].copy()
    pa["launch_speed"] = pd.to_numeric(pa["launch_speed"], errors="coerce")
    pa["hard_hit"]     = (pa["launch_speed"] >= 95).astype(float)
    pa["xwoba"]        = pd.to_numeric(pa["estimated_woba_using_speedangle"], errors="coerce")

    pa_home_off = pa[pa["inning_topbot"]=="Bot"] if "inning_topbot" in pa.columns else pa
    pa_away_off = pa[pa["inning_topbot"]=="Top"] if "inning_topbot" in pa.columns else pa

    def team_last5(side_pa, team_col, prefix):
        g = side_pa.groupby(["game_pk","game_date",team_col]).agg(
            hard_hit=("hard_hit","mean"), xwoba=("xwoba","mean")
        ).reset_index().rename(columns={team_col:"team"})
        g = g.sort_values(["team","game_date"]).reset_index(drop=True)
        g[f"{prefix}_last5_o_hardhit"] = g.groupby("team")["hard_hit"].transform(
            lambda x: x.shift(1).rolling(5,min_periods=1).mean())
        g[f"{prefix}_last5_o_xwoba"]   = g.groupby("team")["xwoba"].transform(
            lambda x: x.shift(1).rolling(5,min_periods=1).mean())
        return g[["game_pk",f"{prefix}_last5_o_hardhit",f"{prefix}_last5_o_xwoba"]]

    home_off = team_last5(pa_home_off, "home_team", "home")
    away_off  = team_last5(pa_away_off, "away_team", "away")
    pa_early  = pa[pa["inning"]<=3].copy() if "inning" in pa.columns else pa.copy()

    def get_starters(side_pa, team_col, side):
        topbot = "Top" if side=="home" else "Bot"
        if "inning_topbot" in side_pa.columns:
            side_pa = side_pa[side_pa["inning_topbot"]==topbot]
        return (
            side_pa.groupby(["game_pk","game_date",team_col,"pitcher"]).size()
            .reset_index(name="n").sort_values("n",ascending=False)
            .groupby("game_pk").first().reset_index()
            [["game_pk","game_date",team_col,"pitcher"]]
        )

    home_starters = get_starters(pa_early, "home_team", "home")
    away_starters = get_starters(pa_early, "away_team", "away")
    p_perf = pa.groupby(["pitcher","game_pk"]).agg(
        p_hard_hit=("hard_hit","mean"), p_xwoba=("xwoba","mean")
    ).reset_index()

    def pitcher_last_start(starters_df, team_col, prefix):
        if starters_df.empty: return pd.DataFrame()
        m = starters_df.merge(p_perf[["game_pk","pitcher","p_hard_hit","p_xwoba"]],
                              on=["game_pk","pitcher"], how="left")
        m = m.sort_values([team_col,"game_date"]).reset_index(drop=True)
        m[f"{prefix}_last_p_hardhit"] = m.groupby(team_col)["p_hard_hit"].shift(1)
        m[f"{prefix}_last_p_xwoba"]   = m.groupby(team_col)["p_xwoba"].shift(1)
        return m[["game_pk",f"{prefix}_last_p_hardhit",f"{prefix}_last_p_xwoba"]]

    home_p  = pitcher_last_start(home_starters, "home_team", "home")
    away_p  = pitcher_last_start(away_starters, "away_team", "away")
    results = _build_win_pct(sc)
    games   = sc.groupby("game_pk").agg(
        game_date=("game_date","first"),
        home_team=("home_team","first"),
        away_team=("away_team","first"),
    ).reset_index()
    gf = games.copy()
    gf = gf.merge(home_off, on="game_pk", how="left")
    gf = gf.merge(away_off,  on="game_pk", how="left")
    if not home_p.empty: gf = gf.merge(home_p, on="game_pk", how="left")
    if not away_p.empty: gf = gf.merge(away_p, on="game_pk", how="left")
    if not results.empty: gf = gf.merge(results, on="game_pk", how="left")
    hist = data.get("game_features", pd.DataFrame())
    if not hist.empty and "home_odds" in hist.columns:
        gf = gf.merge(hist[["game_pk","home_odds","away_odds"]].drop_duplicates("game_pk"),
                      on="game_pk", how="left")
    else:
        gf["home_odds"] = np.nan; gf["away_odds"] = np.nan
    gf["home_odds"] = pd.to_numeric(gf["home_odds"], errors="coerce")
    gf["away_odds"] = pd.to_numeric(gf["away_odds"], errors="coerce")
    gf = gf.sort_values(["game_date","game_pk"]).reset_index(drop=True)
    gf.to_csv(cfg["game_features_master"], index=False)
    data["game_features"] = gf
    print(f"✅ Native game features: {len(gf):,} games")
    return gf

def _dk_fetch_payloads(url):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    caps = DesiredCapabilities.CHROME.copy()
    caps["goog:loggingPrefs"] = {"performance": "ALL"}
    opts.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    driver = webdriver.Chrome(options=opts)
    payloads = []
    try:
        driver.get(url)
        time.sleep(4)
        logs = driver.execute_cdp_cmd("Network.enable", {})
        for entry in driver.get_log("performance"):
            try:
                msg = json.loads(entry["message"])["message"]
                if msg.get("method") == "Network.responseReceived":
                    req_id = msg["params"]["requestId"]
                    resp   = msg["params"]["response"]
                    if "sportscontent" in resp.get("url","") or "sportsbook" in resp.get("url",""):
                        try:
                            body = driver.execute_cdp_cmd("Network.getResponseBody", {"requestId": req_id})
                            parsed = json.loads(body.get("body",""))
                            if parsed.get("events") or parsed.get("markets"): payloads.append(parsed)
                        except: pass
            except: pass
    finally:
        driver.quit()
    return payloads

def scrape_dk_moneyline(target_date=None):
    target = target_date or str(date.today())
    print(f"  Fetching DraftKings moneylines for {target}...")
    try:
        payloads = _dk_fetch_payloads(DK_GAMELINES_URL)
    except Exception as e:
        print(f"  DK moneyline fetch failed: {e}"); return pd.DataFrame()
    if not payloads: print("  No payloads"); return pd.DataFrame()
    events, markets, selections = {}, [], {}
    for d in payloads:
        for e in d.get("events",[]): events[e["id"]] = e
        markets.extend(d.get("markets",[]))
        for s in d.get("selections",[]): selections[s["id"]] = s
    ml_markets = [m for m in markets if "moneyline" in m.get("name","").lower()
                  or "money line" in m.get("name","").lower()]
    rows = []
    for mkt in ml_markets:
        event     = events.get(mkt.get("eventId"),{})
        away_abbr = home_abbr = None
        for p in event.get("participants",[]):
            short = p.get("metadata",{}).get("shortName","") or p.get("name","")
            abbr  = _dk_resolve_name(short)
            if p.get("venueRole")=="Away":   away_abbr = abbr
            elif p.get("venueRole")=="Home": home_abbr = abbr
        if not home_abbr or not away_abbr: continue
        mkt_sels    = [s for s in selections.values() if s.get("marketId")==mkt["id"]]
        sorted_sels = sorted(mkt_sels, key=lambda s: s.get("displayOrder",99))
        if len(sorted_sels) < 2: continue
        away_sel, home_sel = sorted_sels[0], sorted_sels[1]
        home_odds = _dk_american_to_int(home_sel.get("displayOdds",{}).get("american",""))
        away_odds = _dk_american_to_int(away_sel.get("displayOdds",{}).get("american",""))
        if home_odds is None or away_odds is None: continue
        rows.append({"date":target,"home_team":home_abbr,"away_team":away_abbr,
                     "home_odds":home_odds,"away_odds":away_odds})
    if not rows: return pd.DataFrame()
    df = pd.DataFrame(rows)
    print(f"  {len(df)} games scraped")
    return df

def append_dk_moneylines(cfg, target_date=None):
    ml = scrape_dk_moneyline(target_date)
    if ml.empty: return
    gf_path = Path(cfg["game_features_master"])
    if not gf_path.exists(): print("  ⚠️  No game_features_master"); return
    gf = pd.read_csv(gf_path)
    # Normalize game_date to plain string YYYY-MM-DD — avoids all .dt accessor issues
    gf["game_date"] = gf["game_date"].astype(str).str[:10]
    today = str(date.today()) if target_date is None else target_date
    for _, row in ml.iterrows():
        mask = (gf["game_date"] == today) & (gf["home_team"] == row["home_team"])
        if mask.any():
            gf.loc[mask, "home_odds"] = row["home_odds"]
            gf.loc[mask, "away_odds"] = row["away_odds"]
        else:
            new_row = {"game_pk": None, "game_date": today, "home_team": row["home_team"],
                       "away_team": row["away_team"], "home_odds": row["home_odds"], "away_odds": row["away_odds"]}
            gf = pd.concat([gf, pd.DataFrame([new_row])], ignore_index=True)
    gf.to_csv(gf_path, index=False)
    data["game_features"] = gf
    teams = ml["home_team"].tolist()
    print(f"  Moneylines appended for {today}: {teams}")

def build_batting_order_map(cfg):
    if not os.path.exists(NRFI_LINEUP): print(f"⚠️  Not found: {NRFI_LINEUP}"); return
    lu = pd.read_csv(NRFI_LINEUP)
    if "batting_order" not in lu.columns or "player_id" not in lu.columns:
        print("⚠️  Missing columns"); return
    order_map = (
        lu[lu["batting_order"].notna()]
          .groupby("player_id")["batting_order"]
          .apply(lambda x: x.ewm(span=20).mean().iloc[-1])
          .reset_index()
          .rename(columns={"player_id":"batter_id","batting_order":"ewma_batting_order"})
    )
    order_map.to_csv(cfg["player_order_map"], index=False)
    print(f"✅ Batting order map: {len(order_map):,} players")
    return order_map

# Load existing
if Path(cfg["game_features_master"]).exists():
    data["game_features"] = pd.read_csv(cfg["game_features_master"])
    data["game_features"]["game_date"] = pd.to_datetime(data["game_features"]["game_date"])
    print(f"✅ Game features loaded: {len(data['game_features']):,} games")
    if Path(cfg["player_order_map"]).exists():
        om = pd.read_csv(cfg["player_order_map"])
        print(f"✅ Batting order map loaded: {len(om):,} players")
else:
    print("ℹ️  No game features — run build_game_features_native(cfg)")
print("✅ Game features functions defined")

✅ Game features loaded: 12,091 games
✅ Batting order map loaded: 1,524 players
✅ Game features functions defined


## Section 4: Odds Pull + History

In [10]:
def scrape_dk_hr(target_date=None):
    target = target_date or str(date.today())
    print(f"  Fetching DraftKings HR props for {target}...")
    try:
        payloads = _dk_fetch_payloads(DK_HR_URL)
    except Exception as e:
        print(f"  ❌ Fetch failed: {e}"); return []
    if not payloads: print("  ⚠️  No sportscontent payloads"); return []
    events, markets, selections = {}, [], {}
    for d in payloads:
        for e in d.get("events",[]): events[e["id"]] = e
        markets.extend(d.get("markets",[]))
        for s in d.get("selections",[]): selections[s["id"]] = s
    hr_markets = [m for m in markets if str(m.get("name","")).endswith(" Home Runs")]
    if not hr_markets:
        print(f"  ⚠️  No HR markets"); return []
    print(f"  Found {len(hr_markets)} player HR markets")
    results = []
    for mkt in hr_markets:
        event     = events.get(mkt.get("eventId"),{})
        away_abbr = home_abbr = None
        for p in event.get("participants",[]):
            short = p.get("metadata",{}).get("shortName","") or p.get("name","")
            abbr  = _dk_resolve_name(short)
            if p.get("venueRole")=="Away":   away_abbr = abbr
            elif p.get("venueRole")=="Home": home_abbr = abbr
        mkt_sels = [s for s in selections.values() if s.get("marketId")==mkt["id"]]
        anytime  = next((s for s in mkt_sels if s.get("milestoneValue")==1), None)
        if anytime is None:
            mkt_sels_sorted = sorted(mkt_sels, key=lambda s: s.get("milestoneValue",99))
            anytime = mkt_sels_sorted[0] if mkt_sels_sorted else None
        if anytime is None: continue
        hr_yes = _dk_american_to_int(anytime.get("displayOdds",{}).get("american",""))
        if hr_yes is None: continue
        player_name = None
        for p in anytime.get("participants",[]):
            if p.get("type")=="Player": player_name = p.get("name"); break
        if not player_name:
            player_name = mkt.get("name","").replace(" Home Runs","").strip()
        team = None
        for p in anytime.get("participants",[]):
            role = p.get("venueRole","")
            if role=="HomePlayer":   team = home_abbr
            elif role=="AwayPlayer": team = away_abbr
        fair_p = american_to_implied_prob(hr_yes) * 0.93
        results.append({
            "date":player_name,"player_name":player_name,
            "away_team":away_abbr,"home_team":home_abbr,"team":team,
            "consensus_odds":hr_yes,"best_odds":hr_yes,"best_book":"DraftKings",
            "hr_yes":hr_yes,"hr_no":None,
            "fair_p":round(fair_p,4) if pd.notna(fair_p) else np.nan,
            "result":None,
        })
    for r in results: r["date"] = target
    print(f"  ✅ {len(results)} player HR props from DraftKings")
    return results

def fetch_hr_odds_dk():
    results = scrape_dk_hr()
    if not results:
        print("  DK scrape failed — falling back to djstrauss08")
        return fetch_hr_odds_fallback()
    df = pd.DataFrame(results)
    data["odds_today"] = df
    print(f"✅ DraftKings HR props: {len(df)} players | {date.today()}")
    return df

def fetch_hr_odds_fallback():
    url = f"{HR_ODDS_BASE}/players.json"
    r   = _session.get(url, timeout=30)
    r.raise_for_status()
    raw = r.json()
    rows = []
    for p in raw.get("players",[]):
        if p.get("line_display") != "To Hit HR": continue
        over_consensus = p.get("over_odds",{}).get("consensus")
        gc    = p.get("game_context",{})
        books = p.get("over_odds",{}).get("individual_books",[])
        best  = max(books, key=lambda x: x["odds"]) if books else {}
        book_odds = {f"odds_{b['sportsbook'].replace(' ','_').replace('.','').lower()}": b["odds"]
                     for b in books}
        fair_p = american_to_implied_prob(over_consensus) * 0.93 if over_consensus else np.nan
        rows.append({"date":str(date.today()),"player_name":p["player_name"],
                     "away_team":gc.get("away_team"),"home_team":gc.get("home_team"),
                     "consensus_odds":over_consensus,"best_odds":best.get("odds"),
                     "best_book":best.get("sportsbook"),"hr_yes":over_consensus,
                     "hr_no":p.get("under_odds",{}).get("consensus"),
                     "book_count":p.get("sportsbook_count",0),
                     "fair_p":round(fair_p,4) if pd.notna(fair_p) else np.nan,
                     "result":None, **book_odds})
    df = pd.DataFrame(rows)
    data["odds_today"] = df
    print(f"✅ djstrauss08: {len(df)} players")
    return df

def append_all_odds(cfg):
    odds_df = data.get("odds_today", pd.DataFrame())
    if not odds_df.empty:
        path = Path(cfg["odds_master"])
        combined = pd.concat([pd.read_csv(path), odds_df], ignore_index=True) if path.exists() \
                   else odds_df.copy()
        combined.drop_duplicates(subset=["date","player_name"], keep="last", inplace=True)
        combined.to_csv(path, index=False)
        data["odds_master"] = combined
        print(f"  ✅ Odds master: {len(combined):,} rows")

def settle_odds(cfg, settle_date=None):
    if settle_date is None: settle_date = str(date.today()-timedelta(days=1))
    master_path = Path(cfg["odds_master"])
    if not master_path.exists(): print("⚠️  No odds master"); return
    odds = pd.read_csv(master_path)
    sc   = data.get("statcast")
    if sc is None or sc.empty: print("⚠️  No Statcast"); return
    sc["game_date"] = pd.to_datetime(sc["game_date"])
    day_sc    = sc[sc["game_date"].dt.strftime("%Y-%m-%d")==settle_date]
    hr_events = day_sc[day_sc["events"]=="home_run"]
    hr_batter_ids = set(hr_events["batter"].dropna().astype(int))
    lu_path = os.path.join(cfg["base_dir"], f"lineups_{settle_date}.csv")
    lu = pd.read_csv(lu_path) if os.path.exists(lu_path) else data.get("lineups_today", pd.DataFrame())
    if not lu.empty and "batter_name" in lu.columns:
        name_to_id = dict(zip(lu["batter_name"].apply(normalize_name), lu["batter_id"].astype(int)))
    else:
        name_to_id = {}
        print("  ⚠️  No lineup data for settlement")
    mask = odds["date"]==settle_date
    odds.loc[mask,"result"] = odds.loc[mask,"player_name"].apply(
        lambda n: 1 if name_to_id.get(normalize_name(n)) in hr_batter_ids else 0
    )
    odds.to_csv(master_path, index=False)
    data["odds_master"] = odds
    settled = odds[mask]
    print(f"  ✅ Settled {settle_date}: {int(settled['result'].sum())}/{len(settled)} hit HR")

# Load existing
if Path(cfg["odds_master"]).exists():
    data["odds_master"] = pd.read_csv(cfg["odds_master"])
    om = data["odds_master"]
    settled = om[om["result"].notna()]
    print(f"✅ Odds master: {len(om):,} rows | Settled: {len(settled):,}")
else:
    print("ℹ️  No odds master yet")
print("✅ Odds functions defined")

✅ Odds master: 1,704 rows | Settled: 1,194
✅ Odds functions defined


## Section 5: Lineup & Pitcher Pull

In [11]:
def fetch_today_lineups(date_str=None):
    if date_str is None: date_str = str(date.today())
    r = _session.get(MLB_SCHEDULE_URL.format(date=date_str), timeout=30)
    r.raise_for_status()
    games = []
    for d in r.json().get("dates",[]):
        for g in d.get("games",[]):
            games.append({"game_pk":g["gamePk"],
                          "away_name":g["teams"]["away"]["team"]["name"],
                          "home_name":g["teams"]["home"]["team"]["name"]})
    if not games: print(f"No games for {date_str}"); return pd.DataFrame()
    print(f"Found {len(games)} games for {date_str}")
    batters = []
    for g in games:
        try:
            r2  = _session.get(MLB_BOXSCORE_URL.format(game_pk=g["game_pk"]), timeout=30)
            r2.raise_for_status()
            box = r2.json()
        except Exception as e: print(f"  ⚠️  {g['game_pk']}: {e}"); continue
        for side in ["away","home"]:
            team_data     = box.get("teams",{}).get(side,{})
            batting_order = team_data.get("battingOrder",[])
            players       = team_data.get("players",{})
            opp           = "home" if side=="away" else "away"
            for order_idx, player_id in enumerate(batting_order):
                pdata = players.get(f"ID{player_id}",{})
                pinfo = pdata.get("person",{})
                batters.append({
                    "game_pk":g["game_pk"],"game_date":date_str,
                    "batter_id":int(player_id),"batter_name":pinfo.get("fullName"),
                    "batting_order":order_idx+1,"team_side":side,
                    "team_name":g[f"{side}_name"],"opp_name":g[f"{opp}_name"],
                    "home_team":g["home_name"],"away_team":g["away_name"],
                })
    batters_df = pd.DataFrame(batters)
    data["lineups_today"] = batters_df
    if batters_df.empty: print(f"  No lineups posted yet for {date_str}"); return pd.DataFrame()
    cache_path = os.path.join(cfg["base_dir"], f"lineups_{date_str}.csv")
    batters_df.to_csv(cache_path, index=False)
    print(f"  Batters: {len(batters_df)} | Coverage: {batters_df['batter_name'].notna().sum()}/{len(batters_df)}")
    return batters_df

def fetch_probable_pitchers(date_str=None):
    if date_str is None: date_str = str(date.today())
    r = _session.get(MLB_SCHEDULE_PITCHERS_URL.format(date=date_str), timeout=30)
    r.raise_for_status()
    rows = []
    for d in r.json().get("dates",[]):
        for g in d.get("games",[]):
            game_pk = g["gamePk"]
            for side in ["away","home"]:
                team = g["teams"][side]
                prob = team.get("probablePitcher",{})
                opp  = "home" if side=="away" else "away"
                rows.append({"game_pk":game_pk,"game_date":date_str,
                             "pitcher_id":prob.get("id"),"pitcher_name":prob.get("fullName"),
                             "pitcher_side":side,"team_name":team["team"]["name"],
                             "opp_name":g["teams"][opp]["team"]["name"]})
    df    = pd.DataFrame(rows)
    named = df["pitcher_name"].notna().sum() if not df.empty else 0
    print(f"  Probable pitchers: {len(df)} | Named: {named}")
    return df

def build_player_id_map(cfg):
    if not os.path.exists(NRFI_LINEUP): print(f"⚠️  Not found: {NRFI_LINEUP}"); return pd.DataFrame()
    lu = pd.read_csv(NRFI_LINEUP)
    name_map = (
        lu[["player_id","player_name"]]
          .rename(columns={"player_id":"batter_id","player_name":"batter_name"})
          .dropna().drop_duplicates(subset=["batter_id"])
          .assign(name_key=lambda x: x["batter_name"].apply(normalize_name))
    )
    name_map.to_csv(cfg["player_id_map"], index=False)
    print(f"✅ Player ID map: {len(name_map):,} players")
    return name_map

print("✅ Lineup and pitcher functions defined")

✅ Lineup and pitcher functions defined


## Section 6: Player-Game Aggregation (Incremental)

In [12]:
def build_player_game(cfg, statcast_df=None, incremental=True):
    if statcast_df is None: statcast_df = data.get("statcast")
    if statcast_df is None or statcast_df.empty:
        print("⚠️  No Statcast — run Section 1 first"); return pd.DataFrame()

    master_path = Path(cfg["player_game_master"])
    existing, existing_pks = pd.DataFrame(), set()
    if incremental and master_path.exists():
        existing = pd.read_csv(master_path)
        existing["game_date"] = pd.to_datetime(existing["game_date"])
        existing_pks = set(existing["game_pk"].unique())
        print(f"  Existing: {len(existing):,} rows | {len(existing_pks):,} games")

    sc = statcast_df.copy()
    sc["game_date"] = pd.to_datetime(sc["game_date"], infer_datetime_format=True)
    new_sc = sc[~sc["game_pk"].isin(existing_pks)] if incremental else sc

    if new_sc.empty:
        print("  ✅ No new games — player_game up to date")
        data["player_game"] = existing; return existing

    print(f"  Processing {new_sc['game_pk'].nunique()} new games...")
    pa = new_sc[new_sc["events"].notna()].copy()
    pa["hr"]          = (pa["events"]=="home_run").astype(int)
    pa["barrel_flag"] = derive_barrel(pa["launch_speed"],pa["launch_angle"],
                                      pa["launch_speed_angle"] if "launch_speed_angle" in pa.columns else None)
    pa["launch_speed"] = pd.to_numeric(pa["launch_speed"], errors="coerce")
    pa["launch_angle"] = pd.to_numeric(pa["launch_angle"], errors="coerce")
    pa["xwoba"]        = pd.to_numeric(pa["estimated_woba_using_speedangle"], errors="coerce")
    if "inning_topbot" in pa.columns:
        pa["batter_side"] = pa["inning_topbot"].map({"Top":"away","Bot":"home"})
    else:
        pa["batter_side"] = "unknown"

    opp_pitcher = (
        pa.groupby(["batter","game_pk","pitcher"]).size().reset_index(name="pa_vs")
          .sort_values("pa_vs",ascending=False)
          .groupby(["batter","game_pk"]).first().reset_index()
          [["batter","game_pk","pitcher"]].rename(columns={"pitcher":"opp_pitcher_id"})
    )

    if "bb_type" in pa.columns:
        pa["fly_ball"]    = (pa["bb_type"]=="fly_ball").astype(int)
        pa["ground_ball"] = (pa["bb_type"]=="ground_ball").astype(int)
    else:
        pa["fly_ball"]    = (pa["launch_angle"] > 25).fillna(0).astype(int)
        pa["ground_ball"] = (pa["launch_angle"] < 10).fillna(0).astype(int)

    if "hc_x" in pa.columns and "stand" in pa.columns:
        pa["hc_x"]    = pd.to_numeric(pa["hc_x"], errors="coerce")
        pa["pull"]     = np.where(pa["stand"]=="R",(pa["hc_x"]<128).astype(int),(pa["hc_x"]>128).astype(int))
        pa["pull_air"] = ((pa["pull"]==1) & (pa["fly_ball"]==1)).astype(int)
    else:
        pa["pull"] = pa["pull_air"] = 0

    agg_dict = dict(
        game_date         =("game_date",    "first"),
        home_team         =("home_team",    "first"),
        away_team         =("away_team",    "first"),
        stand             =("stand",        "first"),
        batter_side       =("batter_side",  "first"),
        hr                =("hr",           "max"),
        pa                =("hr",           "count"),
        hr_count          =("hr",           "sum"),
        barrel_count      =("barrel_flag",  "sum"),
        max_launch_speed  =("launch_speed", "max"),
        mean_launch_speed =("launch_speed", "mean"),
        mean_launch_angle =("launch_angle", "mean"),
        xwoba_game        =("xwoba",        "mean"),
        fb_count          =("fly_ball",     "sum"),
        gb_count          =("ground_ball",  "sum"),
        pull_air_count    =("pull_air",     "sum"),
    )
    agg = pa.groupby(["batter","game_pk"]).agg(**agg_dict).reset_index()
    agg = agg.merge(opp_pitcher, on=["batter","game_pk"], how="left")

    agg["home_abbr"]      = agg["home_team"].map(TEAM_NAME_TO_ABBR)
    agg["hr_park_factor"] = agg["home_abbr"].map(HR_PARK_FACTORS).fillna(1.0)
    agg["is_outdoor"]     = agg["home_abbr"].map(
        lambda t: 1 if STADIUMS.get(t,(None,None,"open",None))[2]=="open" else 0
    )
    agg["altitude_ft"]    = agg["home_abbr"].map(PARK_ALTITUDE).fillna(0)
    agg["pull_side_dist"] = agg.apply(lambda r: get_pull_side_dist(r.get("home_abbr"), r.get("stand")), axis=1)
    agg["cf_dist"]        = agg["home_abbr"].map(lambda t: PARK_DIMENSIONS.get(t,{}).get("CF",np.nan))

    combined = pd.concat([existing, agg], ignore_index=True)
    combined.drop_duplicates(subset=["batter","game_pk"], keep="last", inplace=True)
    combined = combined.sort_values(["game_date","batter"]).reset_index(drop=True)
    combined.to_csv(master_path, index=False)
    data["player_game"] = combined
    print(f"  ✅ +{len(agg):,} new rows | Total: {len(combined):,} | HR rate: {combined['hr'].mean():.3f}")
    return combined

# Load existing
if Path(cfg["player_game_master"]).exists():
    data["player_game"] = pd.read_csv(cfg["player_game_master"])
    print(f"✅ Player-game loaded: {len(data['player_game']):,} rows")
else:
    print("ℹ️  No player-game — run build_player_game(cfg)")

✅ Player-game loaded: 249,251 rows


## Section 7: Batter Rolling + Platoon Features (Incremental)
Platoon features are the #1 SHAP feature in v5.

**Run:** `build_batter_rolling(cfg)` then `build_platoon_features(cfg)`

**v6 additions:** sweet spot rate, HR zone rate (EV+LA joint), max EV, LA std dev, HR/FB rate, whiff%, chase%, zone contact%

In [13]:
def build_batter_rolling(cfg, statcast_df=None, lookback_days=60):
    if statcast_df is None: statcast_df = data.get("statcast")
    if statcast_df is None or statcast_df.empty:
        print("⚠️  No Statcast — run Section 1 first"); return pd.DataFrame()

    bf_path = Path(cfg["batter_features"])
    cutoff  = pd.Timestamp.today() - pd.Timedelta(days=lookback_days)
    keep    = pd.DataFrame()
    if bf_path.exists():
        existing = pd.read_csv(bf_path)
        existing["game_date"] = pd.to_datetime(existing["game_date"])
        keep = existing[existing["game_date"] < cutoff]
        print(f"  Keeping {len(keep):,} rows outside {lookback_days}d window")

    sc = statcast_df.copy()
    sc["game_date"] = pd.to_datetime(sc["game_date"], infer_datetime_format=True)
    pa = sc[sc["events"].notna()].copy()
    pa["hr"]          = (pa["events"]=="home_run").astype(int)
    pa["barrel"]      = derive_barrel(pa["launch_speed"],pa["launch_angle"],pa.get("launch_speed_angle"))
    pa["launch_speed"]= pd.to_numeric(pa["launch_speed"], errors="coerce")
    pa["hard_hit"]    = (pa["launch_speed"] >= 95).astype(int)
    pa["xwoba"]       = pd.to_numeric(pa["estimated_woba_using_speedangle"], errors="coerce")
    pa["launch_angle"]= pd.to_numeric(pa["launch_angle"], errors="coerce")
    pa["season"]      = pa["game_date"].dt.year

    if "bb_type" in pa.columns:
        pa["fly_ball"]    = (pa["bb_type"]=="fly_ball").astype(int)
        pa["ground_ball"] = (pa["bb_type"]=="ground_ball").astype(int)
    else:
        pa["fly_ball"]    = (pa["launch_angle"] > 25).fillna(0).astype(int)
        pa["ground_ball"] = (pa["launch_angle"] < 10).fillna(0).astype(int)

    if "hc_x" in pa.columns and "stand" in pa.columns:
        pa["hc_x"]    = pd.to_numeric(pa["hc_x"], errors="coerce")
        pa["pull"]     = np.where(pa["stand"]=="R",(pa["hc_x"]<128).astype(int),(pa["hc_x"]>128).astype(int))
        pa["pull_air"] = ((pa["pull"]==1) & (pa["fly_ball"]==1)).astype(int)
    else:
        pa["pull"] = pa["pull_air"] = 0

    # ── v6: new PA-level flags ─────────────────────────────────────────────
    la = pa["launch_angle"]
    ev = pa["launch_speed"]

    # Sweet spot: 8-32 degrees launch angle (contact events only)
    pa["sweet_spot"] = ((la >= 8) & (la <= 32)).astype(int)
    pa["sweet_spot"] = pa["sweet_spot"].where(la.notna(), np.nan)

    # EV+LA joint HR window (true tail event zone)
    pa["hr_zone_contact"] = ((ev >= 95) & (la >= 20) & (la <= 35)).astype(int)
    pa["hr_zone_contact"] = pa["hr_zone_contact"].where(ev.notna() & la.notna(), np.nan)

    # Swing/contact from description + plate location (all pitches needed — see swing_game below)

    # ── Swing metrics from all pitches (not just events) ──────────────────
    sc_all = statcast_df.copy()
    sc_all["game_date"] = pd.to_datetime(sc_all["game_date"], infer_datetime_format=True)
    swing_game = pd.DataFrame(columns=["batter","game_pk"])

    if "description" in sc_all.columns:
        sw = sc_all.copy()
        sw["is_swing"] = sw["description"].isin([
            "swinging_strike","swinging_strike_blocked","foul","foul_tip","hit_into_play"
        ]).astype(int)
        sw["is_whiff"] = sw["description"].isin([
            "swinging_strike","swinging_strike_blocked"
        ]).astype(int)
        if "plate_x" in sw.columns and "plate_z" in sw.columns:
            px  = pd.to_numeric(sw["plate_x"], errors="coerce")
            pz  = pd.to_numeric(sw["plate_z"], errors="coerce")
            szt = pd.to_numeric(sw["sz_top"],  errors="coerce")
            szb = pd.to_numeric(sw["sz_bot"],  errors="coerce")
            sw["in_zone"]    = ((px.abs() <= 0.83) & (pz >= szb) & (pz <= szt)).astype(int)
            sw["chase"]      = ((sw["is_swing"]==1) & (sw["in_zone"]==0)).astype(int)
            sw["zone_swing"] = ((sw["is_swing"]==1) & (sw["in_zone"]==1)).astype(int)
        else:
            sw["in_zone"] = sw["chase"] = sw["zone_swing"] = 0

        swing_game = sw.groupby(["batter","game_pk"]).agg(
            pitches_seen    =("is_swing",   "count"),
            swings          =("is_swing",   "sum"),
            whiffs          =("is_whiff",   "sum"),
            in_zone_pitches =("in_zone",    "sum"),
            chases          =("chase",      "sum"),
            zone_swings     =("zone_swing", "sum"),
        ).reset_index()
        swing_game["swing_pct"]        = swing_game["swings"] / swing_game["pitches_seen"]
        swing_game["whiff_pct"]        = swing_game["whiffs"] / swing_game["swings"].replace(0, np.nan)
        oop = (swing_game["pitches_seen"] - swing_game["in_zone_pitches"]).replace(0, np.nan)
        swing_game["chase_pct"]        = swing_game["chases"] / oop
        swing_game["zone_contact_pct"] = (
            (swing_game["zone_swings"] - swing_game["whiffs"]) /
            swing_game["zone_swings"].replace(0, np.nan)
        )

    # ── Game-level aggregation ─────────────────────────────────────────────
    game_agg = pa.groupby(["batter","game_pk","season"]).agg(
        game_date       =("game_date",        "first"),
        hr_game         =("hr",               "max"),
        barrel_game     =("barrel",           "mean"),
        hard_hit_game   =("hard_hit",         "mean"),
        xwoba_game      =("xwoba",            "mean"),
        ev_game         =("launch_speed",     "mean"),
        ev_max_game     =("launch_speed",     "max"),
        la_mean_game    =("launch_angle",     "mean"),
        la_std_game     =("launch_angle",     "std"),
        fb_game         =("fly_ball",         "mean"),
        gb_game         =("ground_ball",      "mean"),
        pull_air_game   =("pull_air",         "mean"),
        sweet_spot_game =("sweet_spot",       "mean"),
        hr_zone_game    =("hr_zone_contact",  "mean"),
        hr_per_fb_num   =("hr",               "sum"),
        fb_count_game   =("fly_ball",         "sum"),
    ).reset_index()

    game_agg["hr_per_fb_game"] = (
        game_agg["hr_per_fb_num"] / game_agg["fb_count_game"].replace(0, np.nan)
    )

    if not swing_game.empty:
        game_agg = game_agg.merge(
            swing_game[["batter","game_pk","swing_pct","whiff_pct","chase_pct","zone_contact_pct"]],
            on=["batter","game_pk"], how="left"
        )
    else:
        for c in ["swing_pct","whiff_pct","chase_pct","zone_contact_pct"]:
            game_agg[c] = np.nan

    game_agg = game_agg.sort_values(["batter","game_date"]).reset_index(drop=True)

    def roll(series, window):
        return series.shift(1).rolling(window, min_periods=max(1, window // 4)).mean()

    for col, name in [
        ("hr_game",          "batter_hr_rate_L20"),
        ("barrel_game",      "batter_barrel_rate_L20"),
        ("hard_hit_game",    "batter_hard_hit_L20"),
        ("xwoba_game",       "batter_xwoba_L20"),
        ("ev_game",          "batter_launch_speed_L20"),
        ("fb_game",          "batter_fb_rate_L20"),
        ("gb_game",          "batter_gb_rate_L20"),
        ("pull_air_game",    "batter_pull_air_rate_L20"),
        ("hr_game",          "batter_hr_rate_L50"),
        ("barrel_game",      "batter_barrel_rate_L50"),
        # NEW v6
        ("ev_max_game",      "batter_max_ev_L20"),
        ("la_std_game",      "batter_la_std_L20"),
        ("sweet_spot_game",  "batter_sweetspot_rate_L20"),
        ("hr_zone_game",     "batter_hr_zone_rate_L20"),
        ("hr_per_fb_game",   "batter_hr_per_fb_L20"),
        ("whiff_pct",        "batter_whiff_pct_L20"),
        ("chase_pct",        "batter_chase_pct_L20"),
        ("zone_contact_pct", "batter_zone_contact_pct_L20"),
    ]:
        window = 50 if "L50" in name else 20
        game_agg[name] = game_agg.groupby("batter")[col].transform(lambda x: roll(x, window))

    for col, name in [
        ("xwoba_game", "batter_xwoba_season"),
        ("hr_game",    "batter_hr_rate_season"),
    ]:
        game_agg[name] = game_agg.groupby(["batter","season"])[col].transform(
            lambda x: x.shift(1).expanding().mean()
        )

    feat_cols = [
        "batter","game_pk","game_date",
        "batter_hr_rate_L20","batter_barrel_rate_L20","batter_hard_hit_L20",
        "batter_xwoba_L20","batter_launch_speed_L20",
        "batter_fb_rate_L20","batter_gb_rate_L20","batter_pull_air_rate_L20",
        "batter_hr_rate_L50","batter_barrel_rate_L50",
        "batter_xwoba_season","batter_hr_rate_season",
        # NEW v6
        "batter_max_ev_L20","batter_la_std_L20",
        "batter_sweetspot_rate_L20","batter_hr_zone_rate_L20",
        "batter_hr_per_fb_L20",
        "batter_whiff_pct_L20","batter_chase_pct_L20","batter_zone_contact_pct_L20",
    ]
    new_rows = game_agg[[c for c in feat_cols if c in game_agg.columns]][game_agg["game_date"] >= cutoff].copy()
    combined = pd.concat([keep, new_rows], ignore_index=True)
    combined.drop_duplicates(subset=["batter","game_pk"], keep="last", inplace=True)
    combined = combined.sort_values(["game_date","batter"]).reset_index(drop=True)
    combined.to_csv(cfg["batter_features"], index=False)
    data["batter_features"] = combined
    print(f"  ✅ +{len(new_rows):,} recalculated | Total: {len(combined):,} | "
          f"L20 coverage: {combined['batter_hr_rate_L20'].notna().mean():.1%}")
    return combined

# Load existing
if Path(cfg["batter_features"]).exists():
    data["batter_features"] = pd.read_csv(cfg["batter_features"])
    print(f"✅ Batter features loaded: {len(data['batter_features']):,} rows")
else:
    print("ℹ️  No batter features — run build_batter_rolling(cfg)")

✅ Batter features loaded: 243,601 rows


In [14]:
# ── Platoon Features (unchanged from v5) ─────────────────────────────────

def build_platoon_features(cfg, statcast_df=None):
    """
    Batter HR rate vs LHP and vs RHP with Bayesian shrinkage.
    Temporal safety: shift(1) at game level — no lookahead.
    """
    if statcast_df is None: statcast_df = data.get("statcast")
    if statcast_df is None or statcast_df.empty:
        print("⚠️  No Statcast — run Section 1 first"); return pd.DataFrame()

    K_CAREER, K_SEASON = 50, 30
    sc = statcast_df.copy()
    sc["game_date"] = pd.to_datetime(sc["game_date"])
    pa = sc[sc["events"].notna()].copy()
    pa["hr"]     = (pa["events"]=="home_run").astype(int)
    pa["season"] = pa["game_date"].dt.year

    stand_map = pa.groupby("batter")["stand"].first()
    game_hand = (
        pa.groupby(["batter","game_pk","game_date","season","p_throws"])
        .agg(hr_sum=("hr","sum"), pa_sum=("hr","count"))
        .reset_index()
        .sort_values(["batter","p_throws","game_date"])
        .reset_index(drop=True)
    )
    game_hand["stand"] = game_hand["batter"].map(stand_map)

    rates, pas = [], []
    for (batter, p_throws), grp in game_hand.groupby(["batter","p_throws"]):
        cum_hr = grp["hr_sum"].shift(1).expanding().sum()
        cum_pa = grp["pa_sum"].shift(1).expanding().sum()
        rates.extend(cum_hr.values); pas.extend(cum_pa.values)
    game_hand["career_hr_cum"] = rates
    game_hand["career_pa_cum"] = pas

    s_rates = []
    for (batter, p_throws, season), grp in game_hand.groupby(["batter","p_throws","season"]):
        cum_hr = grp["hr_sum"].shift(1).expanding().sum()
        cum_pa = grp["pa_sum"].shift(1).expanding().sum()
        s_rates.extend(list(zip(cum_hr.values, cum_pa.values)))
    game_hand["season_hr_cum"] = [x[0] for x in s_rates]
    game_hand["season_pa_cum"] = [x[1] for x in s_rates]

    def shrink(cum_hr, cum_pa, stand, p_throws, k):
        league = LEAGUE_HR_BY_MATCHUP.get((stand, p_throws), 0.030)
        if pd.isna(cum_hr) or pd.isna(cum_pa) or cum_pa < 1: return league
        return (cum_hr + league * k) / (cum_pa + k)

    game_hand["batter_hr_vs_hand_career"] = game_hand.apply(
        lambda r: shrink(r["career_hr_cum"],r["career_pa_cum"],r["stand"],r["p_throws"],K_CAREER), axis=1
    )
    game_hand["batter_hr_vs_hand_season"] = game_hand.apply(
        lambda r: shrink(r["season_hr_cum"],r["season_pa_cum"],r["stand"],r["p_throws"],K_SEASON), axis=1
    )

    career_pivot = game_hand.pivot_table(
        index=["batter","game_pk","game_date"], columns="p_throws",
        values="batter_hr_vs_hand_career", aggfunc="mean"
    ).reset_index()
    career_pivot.columns.name = None
    career_pivot = career_pivot.rename(columns={"L":"batter_hr_vs_lhp","R":"batter_hr_vs_rhp"})

    season_pivot = game_hand.pivot_table(
        index=["batter","game_pk","game_date"], columns="p_throws",
        values="batter_hr_vs_hand_season", aggfunc="mean"
    ).reset_index()
    season_pivot.columns.name = None
    season_pivot = season_pivot.rename(columns={"L":"batter_hr_vs_lhp_season","R":"batter_hr_vs_rhp_season"})

    out = career_pivot.merge(
        season_pivot[["batter","game_pk","batter_hr_vs_lhp_season","batter_hr_vs_rhp_season"]],
        on=["batter","game_pk"], how="left"
    )
    out.to_csv(cfg["platoon_features"], index=False)
    data["platoon_features"] = out
    print(f"✅ Platoon features: {len(out):,} rows")
    print(f"   vs_lhp: {out['batter_hr_vs_lhp'].notna().mean():.1%} | "
          f"vs_rhp: {out['batter_hr_vs_rhp'].notna().mean():.1%}")
    return out

# Load existing
if Path(cfg["platoon_features"]).exists():
    data["platoon_features"] = pd.read_csv(cfg["platoon_features"])
    print(f"✅ Platoon features loaded: {len(data['platoon_features']):,} rows")
else:
    print("ℹ️  No platoon features — run build_platoon_features(cfg)")

✅ Platoon features loaded: 249,251 rows


## Section 8: Pitcher Rolling Features (Incremental)

**v6 additions:** `pitcher_hr_per_fb_L50`, pitch mix (`pitcher_fb_pct_L50`, `pitcher_brk_pct_L50`, `pitcher_off_pct_L50`, `pitcher_high_fb_pct_L50`), `starter_avg_ip_L5`

In [15]:
def build_pitcher_hr_features(cfg, statcast_df=None, lookback_days=60):
    if statcast_df is None: statcast_df = data.get("statcast")
    if statcast_df is None or statcast_df.empty:
        print("⚠️  No Statcast — run Section 1 first"); return pd.DataFrame()

    pf_path = Path(cfg["pitcher_features"])
    cutoff  = pd.Timestamp.today() - pd.Timedelta(days=lookback_days)
    keep    = pd.DataFrame()
    if pf_path.exists():
        existing = pd.read_csv(pf_path)
        existing["game_date"] = pd.to_datetime(existing["game_date"])
        keep = existing[existing["game_date"] < cutoff]

    sc = statcast_df.copy()
    sc["game_date"] = pd.to_datetime(sc["game_date"], infer_datetime_format=True)

    # ── Events-only (PA level) ─────────────────────────────────────────────
    pa = sc[sc["events"].notna()].copy()
    pa["hr_allowed"]   = (pa["events"]=="home_run").astype(int)
    pa["barrel"]       = derive_barrel(pa["launch_speed"],pa["launch_angle"],pa.get("launch_speed_angle"))
    pa["launch_speed"] = pd.to_numeric(pa["launch_speed"], errors="coerce")
    pa["hard_hit"]     = (pa["launch_speed"] >= 95).astype(int)
    pa["xwoba"]        = pd.to_numeric(pa["estimated_woba_using_speedangle"], errors="coerce")
    pa["launch_angle"] = pd.to_numeric(pa["launch_angle"], errors="coerce")
    pa["fly_ball"]     = (pa["launch_angle"] > 25).fillna(0).astype(int)
    pa["hr_on_fb"]     = ((pa["events"]=="home_run") & (pa["fly_ball"]==1)).astype(int)
    pa["season"]       = pa["game_date"].dt.year
    pa = pa.sort_values(["pitcher","game_date","at_bat_number"]).reset_index(drop=True)

    def roll(series, window):
        return series.shift(1).rolling(window, min_periods=max(1, window // 4)).mean()

    for col, name in [
        ("hr_allowed", "pitcher_hr_rate_L50"),
        ("barrel",     "pitcher_barrel_rate_L50"),
        ("hard_hit",   "pitcher_hard_hit_L50"),
        ("xwoba",      "pitcher_xwoba_L50"),
        ("fly_ball",   "pitcher_fb_rate_L50"),
    ]:
        pa[name] = pa.groupby("pitcher")[col].transform(lambda x: roll(x, 50))

    # HR/FB rate — ratio of two rolling series
    rolled_hr_fb = pa.groupby("pitcher")["hr_on_fb"].transform(lambda x: roll(x, 50))
    rolled_fb    = pa.groupby("pitcher")["fly_ball"].transform(lambda x: roll(x, 50))
    pa["pitcher_hr_per_fb_L50"] = rolled_hr_fb / rolled_fb.replace(0, np.nan)

    # ── All-pitch level: pitch mix + high FB ──────────────────────────────
    sc_p = sc.copy()
    pitch_mix = pd.DataFrame(columns=["pitcher","game_pk"])

    if "pitch_type" in sc_p.columns:
        fastballs = {"FF","SI","FC","FS"}
        breaking  = {"SL","CU","KC","ST","SV","CS"}
        offspeed  = {"CH","SC","KN"}
        sc_p["is_fb"]  = sc_p["pitch_type"].isin(fastballs).astype(int)
        sc_p["is_brk"] = sc_p["pitch_type"].isin(breaking).astype(int)
        sc_p["is_off"] = sc_p["pitch_type"].isin(offspeed).astype(int)

        if "plate_z" in sc_p.columns and "sz_top" in sc_p.columns and "sz_bot" in sc_p.columns:
            pz  = pd.to_numeric(sc_p["plate_z"], errors="coerce")
            szt = pd.to_numeric(sc_p["sz_top"],  errors="coerce")
            szb = pd.to_numeric(sc_p["sz_bot"],  errors="coerce")
            zone_height = (szt - szb).replace(0, np.nan)
            sc_p["high_fb"] = (
                (sc_p["is_fb"] == 1) &
                (pz >= szb + zone_height * 0.67)
            ).astype(int)
        else:
            sc_p["high_fb"] = np.nan

        sc_p = sc_p.sort_values(["pitcher","game_date","at_bat_number"]).reset_index(drop=True)
        for col, name in [
            ("is_fb",   "pitcher_fb_pct_L50"),
            ("is_brk",  "pitcher_brk_pct_L50"),
            ("is_off",  "pitcher_off_pct_L50"),
            ("high_fb", "pitcher_high_fb_pct_L50"),
        ]:
            sc_p[name] = sc_p.groupby("pitcher")[col].transform(lambda x: roll(x, 50))

        pitch_mix = (
            sc_p[["pitcher","game_pk","pitcher_fb_pct_L50","pitcher_brk_pct_L50",
                   "pitcher_off_pct_L50","pitcher_high_fb_pct_L50"]]
            .groupby(["pitcher","game_pk"]).first().reset_index()
        )

    # ── Starter avg IP from outs_when_up ──────────────────────────────────
    starter_ip = pd.DataFrame(columns=["pitcher","game_pk","starter_avg_ip_L5"])
    if "outs_when_up" in sc.columns:
        sc["outs_when_up"] = pd.to_numeric(sc["outs_when_up"], errors="coerce")
        ip_game = (
            sc.groupby(["pitcher","game_pk","game_date"])
              .agg(max_outs=("outs_when_up","max"))
              .reset_index()
        )
        ip_game["ip_game"] = ip_game["max_outs"] / 3.0
        ip_game = ip_game.sort_values(["pitcher","game_date"]).reset_index(drop=True)
        ip_game["starter_avg_ip_L5"] = ip_game.groupby("pitcher")["ip_game"].transform(
            lambda x: x.shift(1).rolling(5, min_periods=1).mean()
        )
        starter_ip = ip_game[["pitcher","game_pk","starter_avg_ip_L5"]]

    # ── Build pitcher-game feature table ──────────────────────────────────
    feat_cols = [
        "pitcher","game_pk","game_date","p_throws",
        "pitcher_hr_rate_L50","pitcher_barrel_rate_L50","pitcher_hard_hit_L50",
        "pitcher_xwoba_L50","pitcher_fb_rate_L50",
        "pitcher_hr_per_fb_L50",
    ]
    pf = (
        pa[[c for c in feat_cols if c in pa.columns]]
          .sort_values(["pitcher","game_pk"])
          .groupby(["pitcher","game_pk"]).first().reset_index()
    )

    if not pitch_mix.empty:
        pf = pf.merge(pitch_mix, on=["pitcher","game_pk"], how="left")
    if not starter_ip.empty:
        pf = pf.merge(starter_ip, on=["pitcher","game_pk"], how="left")

    new_rows = pf[pf["game_date"] >= cutoff]
    combined = pd.concat([keep, new_rows], ignore_index=True)
    combined.drop_duplicates(subset=["pitcher","game_pk"], keep="last", inplace=True)
    combined = combined.sort_values(["game_date","pitcher"]).reset_index(drop=True)
    combined.to_csv(cfg["pitcher_features"], index=False)
    data["pitcher_features"] = combined
    print(f"  ✅ +{len(new_rows):,} recalculated | Total: {len(combined):,} pitcher-game rows")
    return combined

# Load existing
if Path(cfg["pitcher_features"]).exists():
    data["pitcher_features"] = pd.read_csv(cfg["pitcher_features"])
    print(f"✅ Pitcher features loaded: {len(data['pitcher_features']):,} rows")
else:
    print("ℹ️  No pitcher features — run build_pitcher_hr_features(cfg)")

✅ Pitcher features loaded: 103,572 rows


## Section 9: Feature Join
Joins all feature sources into model-ready table.

**v6 changes:** `is_dome` flag, dome temperature fixed to 70°F, handedness park factor join.

In [16]:
def build_features(cfg):
    global features
    pg = data.get("player_game")
    bf = data.get("batter_features")
    pf = data.get("pitcher_features")
    wx = data.get("weather")
    gf = data.get("game_features")

    if pg is None or pg.empty: print("⚠️  No player_game — Section 6"); return None
    if bf is None or bf.empty: print("⚠️  No batter_features — Section 7"); return None

    print(f"Building feature table from {len(pg):,} player-game rows...")
    df = pg.copy()
    df["game_date"] = pd.to_datetime(df["game_date"])

    # Batter join
    bf_c = bf.copy()
    bf_c["game_date"] = pd.to_datetime(bf_c["game_date"])
    drop_cols = [c for c in bf_c.columns if c in df.columns and c not in ["batter","game_pk"]]
    df = df.merge(bf_c.drop(columns=drop_cols), on=["batter","game_pk"], how="left")
    print(f"  After batter join: {len(df):,} | matched: {df['batter_hr_rate_L20'].notna().sum():,}")

    # Pitcher join
    if pf is not None and not pf.empty:
        pf_keyed = pf.rename(columns={"pitcher":"opp_pitcher_id"})
        pcols    = ["opp_pitcher_id","game_pk"] + [
            c for c in pf_keyed.columns if c.startswith("pitcher_") or c in ("p_throws","starter_avg_ip_L5")
        ]
        pf_dedup = pf_keyed[[c for c in pcols if c in pf_keyed.columns]].drop_duplicates(subset=["opp_pitcher_id","game_pk"])
        df = df.merge(pf_dedup, on=["opp_pitcher_id","game_pk"], how="left")
        print(f"  After pitcher join: {df['pitcher_hr_rate_L50'].notna().sum():,} matched "
              f"({df['pitcher_hr_rate_L50'].notna().mean():.1%})")

    # Weather join — v6: add is_dome, fix temperature_f fillna
    if wx is not None and not wx.empty:
        wx_cols = ["game_pk"] + [c for c in wx.columns
                   if c in ["temperature_f","wind_speed_mph","wind_dir_degrees","wind_direction",
                             "precipitation_in","weather_code","wind_out","wind_in",
                             "is_cold","is_hot","high_wind","roof"]]
        df = df.merge(wx[wx_cols], on="game_pk", how="left")
        print(f"  After weather join: {df['temperature_f'].notna().sum():,} matched "
              f"({df['temperature_f'].notna().mean():.1%})")

        # v6: is_dome flag — model can distinguish dome from cold outdoor
        if "roof" in df.columns:
            df["is_dome"] = df["roof"].isin(["dome","retractable"]).astype(int)
        elif "home_abbr" in df.columns:
            df["is_dome"] = df["home_abbr"].map(
                lambda t: 1 if STADIUMS.get(t,(None,None,"open",None))[2] in ("dome","retractable") else 0
            ).fillna(0).astype(int)
        else:
            df["is_dome"] = 0

        # v6: fill temperature_f with 70 for dome/retractable (not 0) to fix air_density
        df["temperature_f"] = df["temperature_f"].fillna(
            df["is_dome"].map({1: 70.0, 0: 0.0})
        ).fillna(0)

        for col in ["wind_speed_mph","wind_out","wind_in","is_cold","is_hot","high_wind"]:
            if col in df.columns: df[col] = df[col].fillna(0)
    else:
        for col in ["temperature_f","wind_speed_mph","wind_out","wind_in","is_cold","is_hot","high_wind"]:
            df[col] = np.nan
        df["is_dome"] = 0

    # Air density — now correct for dome games (temp=70 not 0)
    df["air_density"] = df.apply(
        lambda r: air_density_ratio(r.get("altitude_ft",0), r.get("temperature_f",70)), axis=1
    )

    # v6: Handedness-split park factor join
    df = apply_handedness_park_factor(df)

    # Game features join (team context + moneyline)
    if gf is not None and not gf.empty:
        gf_home = gf[["game_pk","home_last5_o_hardhit","home_last5_o_xwoba",
                       "home_last_p_hardhit","home_last_p_xwoba","home_odds"]].copy() \
                  if "home_last5_o_hardhit" in gf.columns else gf[["game_pk","home_odds"]].copy()
        gf_away = gf[["game_pk","away_last5_o_hardhit","away_last5_o_xwoba",
                       "away_last_p_hardhit","away_last_p_xwoba","away_odds"]].copy() \
                  if "away_last5_o_hardhit" in gf.columns else gf[["game_pk","away_odds"]].copy()

        if "home_last5_o_hardhit" in gf.columns:
            gf_home.columns = ["game_pk","team_last5_hardhit","team_last5_xwoba",
                                "opp_last_start_hardhit","opp_last_start_xwoba","team_moneyline"]
            gf_home["batter_side"] = "home"
            gf_away.columns = ["game_pk","team_last5_hardhit","team_last5_xwoba",
                                "opp_last_start_hardhit","opp_last_start_xwoba","team_moneyline"]
            gf_away["batter_side"] = "away"
            gf_both = pd.concat([gf_home, gf_away], ignore_index=True)
        else:
            gf_home["batter_side"] = "home"
            gf_away.columns = ["game_pk","team_moneyline"]
            gf_away["batter_side"] = "away"
            gf_both = pd.concat([gf_home.rename(columns={"home_odds":"team_moneyline"}),
                                  gf_away], ignore_index=True)

        if "batter_side" in df.columns and df["batter_side"].notna().mean() > 0.5:
            df = df.merge(gf_both, on=["game_pk","batter_side"], how="left")
            null_mask = df["team_moneyline"].isna() if "team_moneyline" in df.columns else pd.Series(False, index=df.index)
            if null_mask.any():
                fill_cols = [c for c in gf_both.columns if c not in ["game_pk","batter_side"]]
                gf_avg = gf_both.groupby("game_pk")[fill_cols].mean().reset_index()
                df.loc[null_mask, fill_cols] = df.loc[null_mask, ["game_pk"]].merge(
                    gf_avg, on="game_pk", how="left"
                )[fill_cols].values
        else:
            df = df.merge(gf_home.drop(columns="batter_side", errors="ignore"), on="game_pk", how="left")

        for col in ["team_last5_hardhit","team_last5_xwoba","opp_last_start_hardhit","opp_last_start_xwoba"]:
            if col in df.columns: df[col] = df[col].fillna(df[col].median())

        df["team_moneyline"]  = pd.to_numeric(df.get("team_moneyline", pd.Series()), errors="coerce")
        df["implied_win_pct"] = df["team_moneyline"].apply(
            lambda x: american_to_implied_prob(x) if pd.notna(x) else np.nan
        )
        print(f"  After game features join: {df['implied_win_pct'].notna().sum():,} matched")
    else:
        df["implied_win_pct"] = np.nan

    # Batting order
    order_path = cfg.get("player_order_map")
    if order_path and os.path.exists(order_path):
        order_map = pd.read_csv(order_path)
        df = df.merge(
            order_map[["batter_id","ewma_batting_order"]].rename(columns={"batter_id":"batter"}),
            on="batter", how="left"
        )
        df["ewma_batting_order"] = df["ewma_batting_order"].fillna(5.0)
        print(f"  After order join: {df['ewma_batting_order'].notna().sum():,} matched")
    else:
        df["ewma_batting_order"] = 5.0

    # Platoon join
    platoon = data.get("platoon_features")
    if platoon is not None and not platoon.empty:
        platoon_c = platoon[["batter","game_pk",
                              "batter_hr_vs_lhp","batter_hr_vs_rhp",
                              "batter_hr_vs_lhp_season","batter_hr_vs_rhp_season"]].copy()
        df = df.merge(platoon_c, on=["batter","game_pk"], how="left")
        if "p_throws" in df.columns:
            df["batter_hr_rate_vs_hand"] = np.where(
                df["p_throws"]=="L", df["batter_hr_vs_lhp"], df["batter_hr_vs_rhp"])
            df["batter_hr_rate_vs_hand_season"] = np.where(
                df["p_throws"]=="L", df["batter_hr_vs_lhp_season"], df["batter_hr_vs_rhp_season"])
        else:
            df["batter_hr_rate_vs_hand"]        = df["batter_hr_vs_rhp"]
            df["batter_hr_rate_vs_hand_season"] = df["batter_hr_vs_rhp_season"]
        df["batter_hr_rate_vs_hand"]        = df["batter_hr_rate_vs_hand"].fillna(0.030)
        df["batter_hr_rate_vs_hand_season"] = df["batter_hr_rate_vs_hand_season"].fillna(0.030)
        print(f"  After platoon join: {df['batter_hr_rate_vs_hand'].notna().mean():.1%} coverage")
    else:
        df["batter_hr_rate_vs_hand"]        = 0.030
        df["batter_hr_rate_vs_hand_season"] = 0.030
        print("  ⚠️  No platoon features — using league average")

    df = df.sort_values("game_date").reset_index(drop=True)
    df.to_csv(cfg["model_features"], index=False)
    features = df

    avail   = [f for f in HR_FEATURES if f in df.columns]
    missing = [f for f in HR_FEATURES if f not in df.columns]
    print(f"  ✅ {len(df):,} rows | Features: {len(avail)}/{len(HR_FEATURES)} | HR rate: {df['hr'].mean():.3f}")
    if missing: print(f"     Missing: {missing}")
    return df

# Load existing
if Path(cfg["model_features"]).exists():
    features = pd.read_csv(cfg["model_features"])
    features["game_date"] = pd.to_datetime(features["game_date"])
    print(f"✅ Features loaded: {len(features):,} rows")
else:
    print("ℹ️  No features — run build_features(cfg)")

✅ Features loaded: 249,251 rows


## Section 10: Model Train (OOS + Walk-forward CV)
Walk-forward CV gives honest AUC estimate. 80/20 OOS for fast iteration.

In [17]:
def train_model(cfg, feat_df=None):
    global model
    if feat_df is None: feat_df = features
    if feat_df is None or feat_df.empty: print("⚠️  No features"); return

    avail = [f for f in HR_FEATURES if f in feat_df.columns]
    print(f"Training on {len(avail)} features")

    df = feat_df.dropna(subset=["hr"]+avail).sort_values("game_date").reset_index(drop=True)
    if len(df) == 0: print("⚠️  No rows after dropna"); return
    split_idx  = int(len(df)*0.80)
    split_date = df.iloc[split_idx]["game_date"]
    train, test = df.iloc[:split_idx], df.iloc[split_idx:]

    print(f"  Train: {len(train):,} through {train['game_date'].max().date()}")
    print(f"  Test:  {len(test):,} from {split_date.date()}")
    print(f"  HR rates: train={train['hr'].mean():.3f} test={test['hr'].mean():.3f}")

    dtrain = xgb.DMatrix(train[avail].astype(float), label=train["hr"].values, feature_names=avail)
    dtest  = xgb.DMatrix(test[avail].astype(float),  label=test["hr"].values,  feature_names=avail)

    m = xgb.train(
        XGB_PARAMS, dtrain, num_boost_round=1000,
        evals=[(dtrain,"train"),(dtest,"test")],
        early_stopping_rounds=50, verbose_eval=100,
    )

    oos_path = cfg["model_xgb"].replace(".json","_oos.json")
    m.save_model(oos_path)
    print(f"  OOS model saved → {oos_path}")

    preds = m.predict(dtest)
    y     = test["hr"].values
    auc   = roc_auc_score(y, preds)
    brier = brier_score_loss(y, preds)

    print(f"\n{'='*50}")
    print(f"OOS RESULTS (test from {split_date.date()})")
    print(f"  AUC:        {auc:.4f}")
    print(f"  Brier:      {brier:.4f}")
    print(f"  Log loss:   {log_loss(y,preds):.4f}")
    print(f"  Cal gap:    {abs(preds.mean()-y.mean()):.4f}")
    print(f"  Best round: {m.best_iteration}")

    imp = pd.Series(m.get_score(importance_type="gain")).sort_values(ascending=False)
    print(f"\nTop features:")
    for f, s in imp.head(10).items(): print(f"  {f:<38} {s:.1f}")

    model["hr"]       = m
    model["features"] = avail
    model["meta"]     = {
        "version":cfg["version"],"auc":auc,"brier":brier,
        "split_date":str(split_date.date()),"best_iteration":m.best_iteration,
        "n_train":len(train),"n_test":len(test),"full_retrain":False,
    }
    return m

def walk_forward_cv(cfg, feat_df=None, scheme="rolling2"):
    """Walk-forward CV. scheme: 'expanding', 'rolling2'. rolling2 recommended."""    
    if feat_df is None: feat_df = features
    avail  = [f for f in HR_FEATURES if f in feat_df.columns]
    df_wf  = feat_df.dropna(subset=["hr"]+avail).copy()
    df_wf["season"] = pd.to_datetime(df_wf["game_date"]).dt.year
    df_wf  = df_wf.sort_values("game_date").reset_index(drop=True)
    szn    = sorted(df_wf["season"].unique())

    if scheme == "expanding":
        folds = [(f"train:{szn[0]}-{szn[i-1]}→test:{szn[i]}",
                  df_wf[df_wf["season"].isin(szn[:i])].index,
                  df_wf[df_wf["season"]==szn[i]].index)
                 for i in range(1, len(szn))]
    else:  # rolling2
        folds = [(f"train:{szn[i-2]}-{szn[i-1]}→test:{szn[i]}",
                  df_wf[df_wf["season"].isin(szn[i-2:i])].index,
                  df_wf[df_wf["season"]==szn[i]].index)
                 for i in range(2, len(szn))]

    print(f"Walk-forward CV ({scheme}) | {len(folds)} folds")
    fold_aucs = []
    for label, tr_idx, te_idx in folds:
        X_tr = df_wf.loc[tr_idx, avail].astype(float)
        X_te = df_wf.loc[te_idx, avail].astype(float)
        y_tr = df_wf.loc[tr_idx, "hr"].values
        y_te = df_wf.loc[te_idx, "hr"].values
        if len(y_te) < 100 or y_te.sum() < 10: continue
        dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=avail)
        dtest  = xgb.DMatrix(X_te, label=y_te, feature_names=avail)
        m = xgb.train(XGB_PARAMS, dtrain, num_boost_round=300,
                      evals=[(dtest,"test")], early_stopping_rounds=30, verbose_eval=False)
        auc = roc_auc_score(y_te, m.predict(dtest))
        cal = abs(m.predict(dtest).mean() - y_te.mean())
        fold_aucs.append(auc)
        print(f"  {label:<45} AUC={auc:.4f}  cal={cal:.4f}  n={len(y_te):,}")
    print(f"\n  Mean AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
    print(f"  This is the honest performance estimate.")
    return fold_aucs

print("✅ train_model and walk_forward_cv defined")

✅ train_model and walk_forward_cv defined


## Section 10b: Full Retrain (Production)

In [18]:
def full_retrain(cfg, feat_df=None):
    global model
    if feat_df is None: feat_df = features
    if feat_df is None or feat_df.empty: print("⚠️  No features"); return
    avail    = [f for f in HR_FEATURES if f in feat_df.columns]
    df       = feat_df.dropna(subset=["hr"]+avail)
    n_rounds = model.get("meta",{}).get("best_iteration",500)
    print(f"Full retrain: {len(df):,} rows | {n_rounds} rounds | {len(avail)} features")
    dtrain = xgb.DMatrix(df[avail].astype(float), label=df["hr"].values, feature_names=avail)
    m = xgb.train(XGB_PARAMS, dtrain, num_boost_round=n_rounds, verbose_eval=False)
    Path(cfg["model_dir"]).mkdir(exist_ok=True)
    m.save_model(cfg["model_xgb"])
    model["hr"]       = m
    model["features"] = avail
    if "meta" not in model: model["meta"] = {}
    # v6: preserve OOS metrics — only update safe keys
    safe_update = {"full_retrain":True,"version":cfg["version"],
                   "n_features":len(avail),"n_rows":len(df)}
    model["meta"].update(safe_update)
    with open(cfg["model_meta"],"w") as f: json.dump(model["meta"],f,indent=2)
    print(f"  ✅ Model saved → {cfg['model_xgb']}")
    return m

def load_model(cfg):
    global model
    if not Path(cfg["model_xgb"]).exists(): print("⚠️  No saved model"); return
    m = xgb.Booster()
    m.load_model(cfg["model_xgb"])
    meta = {}
    if Path(cfg["model_meta"]).exists():
        with open(cfg["model_meta"]) as f: meta = json.load(f)
    feat_df = features if features is not None else (
        pd.read_csv(cfg["model_features"]) if Path(cfg["model_features"]).exists() else None
    )
    avail = [f for f in HR_FEATURES if feat_df is not None and f in feat_df.columns]
    model["hr"]       = m
    model["features"] = avail
    model["meta"]     = meta
    print(f"✅ Model loaded | {meta.get('version','?')} | AUC: {meta.get('auc','N/A')} | "
          f"Features: {len(avail)} | Full retrain: {meta.get('full_retrain')}")

# Auto-load
if Path(cfg["model_xgb"]).exists() and "hr" not in model:
    load_model(cfg)
elif "hr" in model:
    print(f"✅ Model already in memory")
else:
    print("ℹ️  No saved model — run train_model(cfg) then full_retrain(cfg)")

✅ Model loaded | v6 | AUC: 0.6330988508991949 | Features: 22 | Full retrain: True


## Section 11: Calibrate

In [19]:
def evaluate_calibration(cfg, feat_df=None):
    if feat_df is None: feat_df = features
    if "hr" not in model or feat_df is None: print("⚠️  Run Section 10 first"); return
    if model.get("meta",{}).get("full_retrain"):
        print("⚠️  Full retrain model — use OOS model for calibration"); return
    avail      = model["features"]
    split_date = model["meta"]["split_date"]
    df         = feat_df.dropna(subset=["hr"]+avail).sort_values("game_date")
    test       = df[df["game_date"].astype(str)>=split_date]
    dm         = xgb.DMatrix(test[avail].astype(float), feature_names=avail)
    preds      = model["hr"].predict(dm)
    y          = test["hr"].values
    n_bins = 10
    bins   = np.linspace(0, 0.5, n_bins+1)
    bin_cx, act = [], []
    for i in range(n_bins):
        mask = (preds>=bins[i]) & (preds<bins[i+1])
        if mask.sum()>10: bin_cx.append(preds[mask].mean()); act.append(y[mask].mean())
    fig, ax = plt.subplots(figsize=(7,5))
    ax.plot([0,0.4],[0,0.4],"k--",alpha=0.4,label="Perfect")
    ax.scatter(bin_cx, act, s=80, label="Model")
    ax.set_xlabel("Predicted P(HR)"); ax.set_ylabel("Actual HR rate")
    ax.set_title("Reliability Diagram — HR Pro v6")
    ax.legend(); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()
    gap = abs(preds.mean()-y.mean())
    print(f"Mean predicted: {preds.mean():.4f} | Actual: {y.mean():.4f} | Gap: {gap:.4f}")
    print("Gap > 0.02 — run fit_isotonic_calibrator(cfg)" if gap>0.02 else "Calibration OK — skip isotonic")

def fit_isotonic_calibrator(cfg, feat_df=None):
    import pickle
    if feat_df is None: feat_df = features
    if model.get("meta",{}).get("full_retrain"): print("⚠️  Need OOS model"); return
    avail = model["features"]
    df    = feat_df.dropna(subset=["hr"]+avail).sort_values("game_date")
    test  = df[df["game_date"].astype(str)>=model["meta"]["split_date"]]
    dm    = xgb.DMatrix(test[avail].astype(float), feature_names=avail)
    raw   = model["hr"].predict(dm)
    y     = test["hr"].values
    iso   = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw, y)
    with open(cfg["calibrator"],"wb") as f: pickle.dump(iso,f)
    model["calibrator"] = iso
    cal = iso.predict(raw)
    print(f"Brier before: {brier_score_loss(y,raw):.4f} | after: {brier_score_loss(y,cal):.4f}")

print("✅ Calibration functions defined")

✅ Calibration functions defined


## Section 12: Bet Tracker

In [20]:
class HRBetTracker:
    def __init__(self, db_path):
        self.db = db_path
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS bets (
                    id           INTEGER PRIMARY KEY AUTOINCREMENT,
                    date         TEXT,
                    player       TEXT,
                    team         TEXT,
                    vs_pitcher   TEXT,
                    home_team    TEXT,
                    odds         REAL,
                    best_odds    REAL,
                    best_book    TEXT,
                    fair_p       REAL,
                    model_p      REAL,
                    edge         REAL,
                    kelly_pct    REAL,
                    bet_size     REAL,
                    paper        INTEGER DEFAULT 1,
                    result       INTEGER,
                    pnl          REAL,
                    notes        TEXT
                )
            """)

    def log_bet(self, player, odds, fair_p, model_p, edge, kelly_pct, bet_size,
                team=None, vs_pitcher=None, best_odds=None, best_book=None, home_team=None,
                paper=True, notes=""):
        with sqlite3.connect(self.db) as conn:
            conn.execute("""
                INSERT INTO bets
                (date,player,team,vs_pitcher,home_team,odds,best_odds,best_book,
                 fair_p,model_p,edge,kelly_pct,bet_size,paper,notes)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
            """, (str(date.today()),player,team,vs_pitcher,home_team,odds,best_odds,best_book,
                  fair_p,model_p,edge,kelly_pct,bet_size,int(paper),notes))
        print(f"  Logged: {player} | +{odds} | edge {edge:+.0%} | ${bet_size:.0f} | paper={paper}")

    def settle_by_date(self, settle_date=None):
        if settle_date is None: settle_date = str(date.today()-timedelta(days=1))
        sc = data.get("statcast")
        if sc is None or sc.empty: print("⚠️  No Statcast"); return
        sc_day    = sc[pd.to_datetime(sc["game_date"]).dt.strftime("%Y-%m-%d")==settle_date]
        hr_events = sc_day[sc_day["events"]=="home_run"]
        hr_batter_ids = set(hr_events["batter"].dropna().astype(int))
        lu_path = os.path.join(cfg["base_dir"], f"lineups_{settle_date}.csv")
        lu = pd.read_csv(lu_path) if os.path.exists(lu_path) else data.get("lineups_today",pd.DataFrame())
        if not lu.empty and "batter_name" in lu.columns:
            name_to_id = dict(zip(lu["batter_name"].apply(normalize_name), lu["batter_id"].astype(int)))
        else:
            name_to_id = {}
            print("  ⚠️  No lineup cache for settlement")
        with sqlite3.connect(self.db) as conn:
            pending = conn.execute(
                "SELECT id,player,odds,bet_size FROM bets WHERE date=? AND result IS NULL",
                (settle_date,)
            ).fetchall()
        if not pending: print(f"No pending bets for {settle_date}"); return
        for bet_id, player, odds, bet_size in pending:
            batter_id = name_to_id.get(normalize_name(player))
            result    = 1 if batter_id and batter_id in hr_batter_ids else 0
            pnl       = bet_size*(odds/100 if odds>0 else 100/abs(odds)) if result else -bet_size
            with sqlite3.connect(self.db) as conn:
                conn.execute("UPDATE bets SET result=?,pnl=? WHERE id=?", (result,pnl,bet_id))
            print(f"  {player:<25} {'WIN ✅' if result else 'LOSS'} | P&L: ${pnl:+.2f}")
        print(f"  Settled {len(pending)} bets for {settle_date}")

    def dashboard(self):
        with sqlite3.connect(self.db) as conn:
            df = pd.read_sql("SELECT * FROM bets ORDER BY date DESC, id DESC", conn)
        if df.empty: print("No bets logged yet"); return df
        settled = df[df["result"].notna()]
        paper   = settled[settled["paper"]==1]
        real    = settled[settled["paper"]==0]
        print(f"\n{'='*55}")
        print(f"HR PRO v6 — BET TRACKER")
        print(f"{'='*55}")
        print(f"  Total: {len(df)} | Settled: {len(settled)} | Pending: {len(df)-len(settled)}")
        for label, subset in [("Paper",paper),("Real",real)]:
            if len(subset)==0: continue
            hr    = subset["result"].mean()
            pnl   = subset["pnl"].sum()
            vol   = subset["bet_size"].sum()
            roi   = pnl/vol*100 if vol>0 else 0
            avg_o = subset["odds"].mean()
            be    = american_to_implied_prob(avg_o)
            print(f"\n  [{label}] {len(subset)} bets")
            print(f"    Hit rate:  {hr:.1%} (breakeven: {be:.1%} at avg +{avg_o:.0f})")
            print(f"    P&L:       ${pnl:+.2f} | ROI: {roi:+.1f}%")
        return df

tracker = HRBetTracker(cfg["bet_db"])
print(f"✅ Bet tracker initialized | {cfg['bet_db']}")

✅ Bet tracker initialized | C:\Users\lmayn\Downloads\HR_Pro\data\hr_bets_v6.db


## Section 13: Daily Command Center

**Morning run (once):** Run the full cell — pulls data, settles yesterday, rebuilds features, generates signals.  
**Throughout the day:** Run `picks = refresh_signals()` to re-pull latest odds without rebuilding features.

In [21]:
def generate_signals(cfg, feat_df=None, bankroll=1000.0):
    """Generate today's HR prop signals."""
    global features
    if feat_df is None:
        if features is None:
            features = pd.read_csv(cfg["model_features"])
            features["game_date"] = pd.to_datetime(features["game_date"])
        feat_df = features
    if "hr" not in model: print("⚠️  No model — run Section 10b"); return pd.DataFrame()

    odds_df     = fetch_hr_odds_dk()
    pitchers_df = fetch_probable_pitchers()
    wx_today    = weather_live(cfg)
    if odds_df.empty: print("⚠️  No odds"); return pd.DataFrame()

    # Player ID map
    id_map_path = cfg["player_id_map"]
    if os.path.exists(id_map_path):
        id_map            = pd.read_csv(id_map_path)
        name_to_batter_id = dict(zip(id_map["name_key"], id_map["batter_id"].astype(int)))
    else:
        name_to_batter_id = {}
        print("  ⚠️  No player_id_map — run build_player_id_map(cfg)")

    # Pitcher lookup
    pitcher_by_team = {}
    if not pitchers_df.empty:
        for _, row in pitchers_df.iterrows():
            if pd.notna(row.get("pitcher_id")):
                opp_abbr = TEAM_NAME_TO_ABBR.get(row["opp_name"], row["opp_name"])
                pitcher_by_team[opp_abbr] = {
                    "opp_pitcher_id":   int(row["pitcher_id"]),
                    "opp_pitcher_name": row["pitcher_name"],
                }

    # Weather lookup
    wx_by_home = {}
    if not wx_today.empty:
        for _, row in wx_today.iterrows():
            wx_by_home[row.get("home_team")] = row

    avail       = model["features"]
    feat_latest = feat_df.sort_values("game_date").groupby("batter").last().reset_index()
    pf          = data.get("pitcher_features", pd.DataFrame())
    pf_latest   = pf.sort_values("game_date").groupby("pitcher").last().reset_index() \
                  if not pf.empty else pd.DataFrame()

    # Batting order map
    order_path   = cfg.get("player_order_map","")
    order_lookup = {}
    if os.path.exists(order_path):
        om = pd.read_csv(order_path)
        order_lookup = dict(zip(om["batter_id"].astype(int), om["ewma_batting_order"]))

    # Platoon lookup
    platoon_df     = data.get("platoon_features", pd.DataFrame())
    platoon_latest = platoon_df.sort_values("game_date").groupby("batter").last().reset_index() \
                    if not platoon_df.empty else pd.DataFrame()

    lu       = data.get("lineups_today", pd.DataFrame())
    results  = []
    no_match = []

    for _, odds_row in odds_df.iterrows():
        player_name = odds_row["player_name"]
        team        = odds_row.get("team")
        home_team   = odds_row.get("home_team")

        # Resolve batter_id
        batter_id = None
        if not lu.empty and "batter_name" in lu.columns:
            lu_match = lu[lu["batter_name"].apply(normalize_name)==normalize_name(player_name)]
            if not lu_match.empty: batter_id = int(lu_match.iloc[0]["batter_id"])
        if batter_id is None:
            batter_id = name_to_batter_id.get(resolve_dk_name(player_name))
        if batter_id is None: no_match.append(player_name); continue

        brow = feat_latest[feat_latest["batter"]==batter_id]
        if brow.empty: print(f"  ⚠️  No features: {player_name}"); continue

        x = brow.iloc[0].copy()

        # Override pitcher features
        pitcher_info = pitcher_by_team.get(team, {})
        opp_pid      = pitcher_info.get("opp_pitcher_id")
        p_throws     = None
        if not pf_latest.empty and opp_pid:
            prow = pf_latest[pf_latest["pitcher"]==opp_pid]
            if not prow.empty:
                for col in [c for c in avail if c.startswith("pitcher_") or c=="starter_avg_ip_L5"]:
                    if col in prow.columns: x[col] = prow.iloc[0][col]
                if "p_throws" in prow.columns:
                    p_throws = prow.iloc[0]["p_throws"]

        # Override platoon based on today's pitcher hand
        if p_throws is not None and not platoon_latest.empty:
            prow_p = platoon_latest[platoon_latest["batter"]==batter_id]
            if not prow_p.empty:
                col_c = "batter_hr_vs_lhp" if p_throws=="L" else "batter_hr_vs_rhp"
                col_s = "batter_hr_vs_lhp_season" if p_throws=="L" else "batter_hr_vs_rhp_season"
                if col_c in prow_p.columns and "batter_hr_rate_vs_hand" in avail:
                    x["batter_hr_rate_vs_hand"]        = prow_p.iloc[0][col_c]
                    x["batter_hr_rate_vs_hand_season"] = prow_p.iloc[0][col_s]

        # Override weather + air density
        home_abbr = TEAM_NAME_TO_ABBR.get(str(home_team), str(home_team))
        wx_row    = wx_by_home.get(home_abbr, {})
        for wx_col in ["temperature_f","wind_speed_mph","wind_out","wind_in",
                        "is_cold","is_hot","high_wind"]:
            if wx_col in wx_row and pd.notna(wx_row.get(wx_col)) and wx_col in avail:
                x[wx_col] = wx_row[wx_col]

        # is_dome override
        if "is_dome" in avail:
            roof = STADIUMS.get(home_abbr, (None,None,"open",None))[2]
            x["is_dome"] = 1 if roof in ("dome","retractable") else 0

        if "air_density" in avail:
            alt = PARK_ALTITUDE.get(home_abbr, 0)
            tmp = wx_row.get("temperature_f", 70) or 70
            x["air_density"] = air_density_ratio(alt, tmp)

        # hr_park_factor_hand override
        if "hr_park_factor_hand" in avail:
            pf_today = data.get("park_factors", pd.DataFrame())
            stand_val = x.get("stand")
            if not pf_today.empty and stand_val:
                cur_season = date.today().year
                max_pf_szn = pf_today["season"].max()
                lookup_szn = min(cur_season, max_pf_szn)
                match = pf_today[(pf_today["team_abbr"]==home_abbr) &
                                  (pf_today["season"]==lookup_szn) &
                                  (pf_today["stand"]==stand_val)]
                if not match.empty:
                    x["hr_park_factor_hand"] = match.iloc[0]["hr_factor"]

        # Override batting order
        if batter_id in order_lookup and "ewma_batting_order" in avail:
            x["ewma_batting_order"] = order_lookup[batter_id]

        feat_vec = pd.DataFrame([x[avail]])
        for c in feat_vec.columns: feat_vec[c] = pd.to_numeric(feat_vec[c], errors="coerce")

        dm   = xgb.DMatrix(feat_vec, feature_names=avail)
        p_hr = calibrate(float(model["hr"].predict(dm)[0]))
        edge = p_hr - odds_row["fair_p"]

        exp_pa   = float(x.get("ewma_batting_order", 5.0))
        p_2hr    = p_two_plus_hr(p_hr, avg_pa=order_to_expected_pa(exp_pa))
        kelly    = kelly_stake(edge, odds_row["consensus_odds"], cfg["kelly_fraction"])
        kelly_pct= min(max(kelly,0), cfg["max_kelly_pct"])
        bet_size = round(kelly_pct*bankroll,2) if kelly_pct>=cfg["min_kelly_pct"] else 0

        results.append({
            "player":         player_name,
            "team":           team,
            "home_team":      home_team,
            "vs_pitcher":     pitcher_info.get("opp_pitcher_name"),
            "pitcher_hand":   p_throws,
            "consensus_odds": odds_row["consensus_odds"],
            "best_odds":      odds_row["best_odds"],
            "best_book":      odds_row.get("best_book"),
            "fair_p":         round(odds_row["fair_p"],4),
            "model_p":        round(p_hr,4),
            "model_p_2hr":    round(p_2hr,4),
            "edge":           round(edge,4),
            "kelly_pct":      round(kelly_pct,4),
            "bet_size":       bet_size,
            "signal":         1 if kelly_pct >= cfg["min_kelly_pct"] and edge > 0 else 0,
        })

    if no_match:
        print(f"  ⚠️  No ID match for {len(no_match)} players: {no_match}")
    if not results:
        print("No results — check player_id_map coverage")
        return pd.DataFrame()

    df = pd.DataFrame(results).sort_values("edge",ascending=False).reset_index(drop=True)
    return df

print("✅ generate_signals defined")

✅ generate_signals defined


In [22]:
def daily_dashboard(signals_df, cfg, bankroll=1000.0, max_per_game=2):
    """Display today's picks. v6: team column, whole-number formatting."""    
    if signals_df is None or signals_df.empty: print("No signals to display"); return

    today_str = str(date.today())
    signals   = signals_df[signals_df["signal"]==1].copy()
    all_df    = signals_df.copy()

    signals["game_group"] = signals["vs_pitcher"].fillna("unknown")
    game_counts           = signals["game_group"].value_counts()

    # Cap per game
    signals_capped = (
        signals.sort_values("edge",ascending=False)
               .groupby("game_group").head(max_per_game)
               .sort_values("edge",ascending=False).reset_index(drop=True)
    )
    dropped = len(signals) - len(signals_capped)

    # Correlation Kelly scaling
    signals_capped["adj_kelly_pct"] = signals_capped.apply(
        lambda r: r["kelly_pct"] / (game_counts.get(r["game_group"],1)**0.5), axis=1
    )
    signals_capped["adj_bet_size"] = signals_capped["adj_kelly_pct"].apply(
        lambda k: round(k*bankroll,2) if k>=cfg["min_kelly_pct"] else 0
    )

    print()
    print("╔" + "═"*67 + "╗")
    print(f"║  HR PRO v6 — {today_str}  |  Bankroll: ${bankroll:.0f}{'':>22}║")
    if "hr" in model:
        meta = model.get("meta",{})
        try:
            auc_str = f"{meta.get('auc','?'):.4f}"
        except:
            auc_str = str(meta.get('auc','?'))
        print(f"║  Model: {meta.get('version','?')} | AUC: {auc_str} | "
              f"{'full retrain' if meta.get('full_retrain') else 'OOS eval'}{'':>24}║"[:70]+"║")
    print("╚" + "═"*67 + "╝")

    n_eval    = len(all_df)
    n_pos     = (all_df["edge"]>0).sum()
    mean_edge = all_df["edge"].mean()
    print(f"\n  Evaluated: {n_eval} | Edge>0: {n_pos} | Edge≤0: {n_eval-n_pos} | Mean: {mean_edge:+.0%}")

    # Correlation warnings
    correlated = game_counts[game_counts>1]
    for pitcher, count in correlated.items():
        print(f"  ⚠️  {count}x vs {pitcher} — Kelly scaled 1/√{count}")
    if dropped > 0:
        print(f"  ℹ️  {dropped} signal(s) dropped (cap={max_per_game}/game)")

    # Bets table — v6: Team column (not pitcher), whole-number edge + bet size
    if signals_capped.empty:
        print("\n  No signals today")
    else:
        print(f"\n  {'BETS':─<65}")
        print(f"  {'Player':<22} {'Team':<6} {'Odds':>6} {'Fair':>6} {'Model':>6} {'Edge':>6} {'Bet$':>5} {'2+HR':>6}")
        print(f"  {'─'*22} {'─'*6} {'─'*6} {'─'*6} {'─'*6} {'─'*6} {'─'*5} {'─'*6}")
        for _, r in signals_capped.iterrows():
            corr_tag = f" [×{game_counts.get(r['game_group'],1)}]" if game_counts.get(r['game_group'],1)>1 else "      "
            print(f"  {r['player']:<22} {str(r.get('team','?')):<6} "
                  f"{r['consensus_odds']:>+6} {r['fair_p']:>5.1%} {r['model_p']:>5.1%} "
                  f"{r['edge']:>+5.0%} ${r['adj_bet_size']:>3.0f} {r['model_p_2hr']:>4.1%}{corr_tag}")
        total_exp = signals_capped["adj_bet_size"].sum()
        print(f"  {'─'*65}")
        print(f"  {'':>58} Total exposure ${total_exp:.0f}  ({total_exp/bankroll:.1%})")

    # 2+ HR watch
    high_2hr = all_df[all_df["model_p_2hr"] >= 0.025].sort_values("model_p_2hr",ascending=False).head(5)
    if not high_2hr.empty:
        print(f"\n  {'2+ HR WATCH':─<65}")
        for _, r in high_2hr.iterrows():
            print(f"  {r['player']:<22} {str(r.get('team','?')):<6} "
                  f"1HR: {r['model_p']:.1%}  2+HR: {r['model_p_2hr']:.1%}  (check DK 2+ line)")

    # Near misses
    near_miss = all_df[
        (all_df["edge"]>=cfg["min_edge"]*0.5) &
        (all_df["edge"]< cfg["min_edge"]) &
        (all_df["signal"]==0)
    ].sort_values("edge",ascending=False).head(5)
    if not near_miss.empty:
        print(f"\n  {'NEAR MISSES':─<65}")
        for _, r in near_miss.iterrows():
            print(f"  {r['player']:<22} {str(r.get('team','?')):<6} "
                  f"+{int(r['consensus_odds'])}  edge: {r['edge']:>+4.0%}")

    # Weather
    wx = data.get("weather_today", pd.DataFrame())
    if not wx.empty:
        outdoor = wx[wx["is_outdoor"]==1]
        if not outdoor.empty:
            print(f"\n  {'WEATHER':─<65}")
            for _, w in outdoor.iterrows():
                temp  = f"{w['temperature_f']:.0f}°F" if pd.notna(w.get("temperature_f")) else "N/A"
                wind  = f"{w['wind_speed_mph']:.0f}mph {w.get('wind_direction','')}" if pd.notna(w.get("wind_speed_mph")) else "N/A"
                flags = []
                if w.get("wind_out")==1:  flags.append("OUT")
                if w.get("wind_in")==1:   flags.append("IN")
                if w.get("is_hot")==1:    flags.append("HOT")
                if w.get("is_cold")==1:   flags.append("COLD")
                print(f"  {w.get('home_team','?')} vs {w.get('away_team','?'):<6} "
                      f"Temp:{temp:<8} Wind:{wind:<14} {' '.join(flags)}")

    # Performance
    print(f"\n  {'SEASON PERFORMANCE':─<65}")
    tracker.dashboard()

    return signals_capped

def refresh_signals(bankroll=None):
    """Re-pull odds and regenerate signals without rebuilding features."""    
    global features
    if bankroll is None: bankroll = BANKROLL
    if features is None:
        features = pd.read_csv(cfg["model_features"])
        features["game_date"] = pd.to_datetime(features["game_date"])
    return daily_dashboard(
        signals_df=generate_signals(cfg, bankroll=bankroll),
        cfg=cfg, bankroll=bankroll, max_per_game=MAX_PER_GAME
    )

print("✅ daily_dashboard and refresh_signals defined")

✅ daily_dashboard and refresh_signals defined


In [23]:
import pandas as pd

# Monkey-patch to fix deprecated argument
_orig_to_datetime = pd.to_datetime
def _patched_to_datetime(*args, **kwargs):
    kwargs.pop("infer_datetime_format", None)
    return _orig_to_datetime(*args, **kwargs)
pd.to_datetime = _patched_to_datetime
print("✅ pd.to_datetime patched")

# Fix game_pk dtype mismatch
if "game_pk" in data["weather"].columns:
    data["weather"]["game_pk"] = pd.to_numeric(data["weather"]["game_pk"], errors="coerce").astype("Int64")
print("✅ game_pk fixed")

✅ pd.to_datetime patched
✅ game_pk fixed


In [24]:
# ╔══════════════════════════════════════════════════════════════╗
# ║         HR PRO v6 — DAILY COMMAND CENTER                    ║
# ║  Run this cell every morning. Use refresh_signals() later.  ║
# ╚══════════════════════════════════════════════════════════════╝

BANKROLL     = 1000.0   # update as needed
PAPER        = True     # set False when ready for real money
MAX_PER_GAME = 2        # max correlated bets on same pitcher

print(f"{'='*65}")
print(f"  HR PRO v6 — DAILY COMMAND CENTER — {date.today()}")
print(f"{'='*65}")

# STEP 1: Nightly maintenance
print("\n── STEP 1: NIGHTLY MAINTENANCE")
statcast_nightly(cfg)
weather_nightly(cfg)

# STEP 2: Settle yesterday
print("\n── STEP 2: SETTLE YESTERDAY")
tracker.settle_by_date()
#settle_odds(cfg)

# STEP 3: Rebuild features (incremental)
print("\n── STEP 3: REBUILD FEATURES")
build_player_game(cfg, incremental=True)
build_batter_rolling(cfg)
build_platoon_features(cfg)
build_pitcher_hr_features(cfg)
append_dk_moneylines(cfg)
data["weather"] = pd.read_csv(cfg["weather_master"])
data["weather"]["game_date"] = pd.to_datetime(data["weather"]["game_date"])
build_features(cfg)

# ── Upload features to GCS ────────────────────────────────────────────────
import subprocess
result = subprocess.run([
    r"C:\Program Files (x86)\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.cmd",
    "storage", "cp",
    cfg["model_features"],
    "gs://concrete-crow-445205-m4-mlb-data/HR_Pro/data/model_features.csv"
], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ model_features.csv uploaded to GCS")
else:
    print(f"⚠️  GCS upload failed: {result.stderr}")


# STEP 4: Generate signals
print("\n── STEP 4: TODAY'S SIGNALS")
signals_df = generate_signals(cfg, bankroll=BANKROLL)

# STEP 5: Dashboard + log
print("\n── STEP 5: DASHBOARD")
picks = daily_dashboard(signals_df, cfg, bankroll=BANKROLL, max_per_game=MAX_PER_GAME)

# STEP 6: Save
print("\n── STEP 6: SAVE")
if picks is not None and not picks.empty:
    for _, r in picks[picks["adj_bet_size"]>0].iterrows():
        tracker.log_bet(
            player=r["player"], odds=r["consensus_odds"],
            fair_p=r["fair_p"], model_p=r["model_p"], edge=r["edge"],
            kelly_pct=r["adj_kelly_pct"], bet_size=r["adj_bet_size"],
            team=r.get("team"), vs_pitcher=r.get("vs_pitcher"),
            best_odds=r.get("best_odds"), best_book=r.get("best_book"),
            home_team=r.get("home_team"), paper=PAPER,
        )
fetch_hr_odds_fallback()
append_all_odds(cfg)

  HR PRO v6 — DAILY COMMAND CENTER — 2026-05-07

── STEP 1: NIGHTLY MAINTENANCE
Statcast nightly: 2026-05-06
  No data for yesterday
  Files: 926 with data | 320 empty
  ✅ 926,980 rows | 2021-04-01 → 2026-04-14
Weather nightly: 2026-05-06
  ✅ 15 games
  ✅ Weather: 12,334 games

── STEP 2: SETTLE YESTERDAY
  ⚠️  No lineup cache for settlement
No pending bets for 2026-05-06

── STEP 3: REBUILD FEATURES
  Existing: 249,251 rows | 12,301 games
  ✅ No new games — player_game up to date
  Keeping 240,500 rows outside 60d window
  ✅ +3,101 recalculated | Total: 243,601 | L20 coverage: 97.2%
✅ Platoon features: 249,251 rows
   vs_lhp: 52.2% | vs_rhp: 91.2%
  ✅ +1,321 recalculated | Total: 103,572 pitcher-game rows
  Fetching DraftKings moneylines for 2026-05-07...
  10 games scraped
  Moneylines appended for 2026-05-07: ['NYY', 'WAS', 'KC', 'CHC', 'COL', 'ARI', 'MIA', 'PHI', 'BOS', 'SD']
Building feature table from 249,251 player-game rows...
  After batter join: 249,251 | matched: 236,781
  A

In [25]:
# ── INTRADAY REFRESH — run any time during the day ───────────────────────
# Re-pulls latest DK odds and regenerates signals. Does NOT rebuild features.
picks = refresh_signals()

  Fetching DraftKings HR props for 2026-05-07...
  ⚠️  No HR markets
  DK scrape failed — falling back to djstrauss08
✅ djstrauss08: 14 players
  Probable pitchers: 20 | Named: 20
Live weather: 2026-05-07
home_team away_team  temperature_f  wind_speed_mph wind_direction  wind_out  wind_in
      NYY       TEX           58.8             8.9             NW         0        0
      WSH       MIN           58.3             0.9              W         0        0
       KC       CLE           64.4            15.7             SW         0        0
      CHC       CIN           57.9            10.8             SW         1        1
      COL       NYM           63.9             1.9              W         0        0
      PHI       OAK           64.2             8.5             NW         0        0
      BOS        TB           58.3            12.1              W         0        1
       SD       STL           65.1             8.4             NW         0        0
  ⚠️  No ID match for 1 player

## Section 14: Mathematical Rigor Assessment
Run after any retrain. Requires OOS model file.

In [26]:
try:
    import shap
except ImportError:
    import subprocess; subprocess.run(["pip","install","shap","--quiet"]); import shap

from scipy import stats
from scipy.stats import chi2, binom
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.calibration import calibration_curve
import matplotlib.gridspec as gridspec

print("=" * 70)
print("  HR PRO v6 — MATHEMATICAL RIGOR ASSESSMENT")
print("=" * 70)

if features is None or features.empty:
    features = pd.read_csv(cfg["model_features"])
    features["game_date"] = pd.to_datetime(features["game_date"])
if "hr" not in model:
    print("⚠️  Run Section 10 first"); raise SystemExit

avail = [f for f in model.get("features", HR_FEATURES) if f in features.columns]
df    = features.dropna(subset=["hr"]+avail).sort_values("game_date").reset_index(drop=True)

split_idx  = int(len(df)*0.80)
X          = df[avail].astype(float)
y          = df["hr"].values
X_test     = X.iloc[split_idx:].copy()
y_test     = y[split_idx:]
n_test     = len(y_test)
base_rate  = y_test.mean()

# Load OOS model
oos_path = cfg["model_xgb"].replace(".json","_oos.json")
if Path(oos_path).exists():
    booster_eval = xgb.Booster()
    booster_eval.load_model(oos_path)
    print(f"✅ OOS model loaded")
else:
    print("⚠️  No OOS model — using in-memory model")
    booster_eval = model["hr"]

dtest  = xgb.DMatrix(X_test, feature_names=avail)
p_test = booster_eval.predict(dtest)

# 1. Core metrics
auc    = roc_auc_score(y_test, p_test)
prauc  = average_precision_score(y_test, p_test)
brier  = brier_score_loss(y_test, p_test)
ll     = log_loss(y_test, p_test)
brier_baseline = brier_score_loss(y_test, np.full(n_test, base_rate))
ll_baseline    = log_loss(y_test, np.full(n_test, base_rate))
brier_skill    = 1 - brier / brier_baseline
prauc_skill    = (prauc - base_rate) / (1 - base_rate)

print(f"\n── CORE METRICS")
print(f"  Test rows:    {n_test:,} | Base HR rate: {base_rate:.3f}")
print(f"  AUC:          {auc:.4f}")
print(f"  PR-AUC:       {prauc:.4f}  (baseline={base_rate:.3f})  skill={prauc_skill:+.4f}")
print(f"  Brier skill:  {brier_skill:+.4f}")
print(f"  Cal gap:      {abs(p_test.mean()-base_rate):.4f}")

# 2. AUC CI
n1 = int(y_test.sum()); n0 = n_test-n1
q1 = auc/(2-auc); q2 = 2*auc**2/(1+auc)
se = np.sqrt((auc*(1-auc)+(n1-1)*(q1-auc**2)+(n0-1)*(q2-auc**2))/(n1*n0))
ci_lo, ci_hi = auc-1.96*se, auc+1.96*se
z_auc = (auc-0.5)/se
p_auc = 2*(1-stats.norm.cdf(abs(z_auc)))
print(f"\n── AUC CONFIDENCE INTERVAL")
print(f"  95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]  p={p_auc:.2e}  "
      f"{'significant ✅' if p_auc<0.05 else 'not significant ⚠️'}")

# 3. Calibration (Hosmer-Lemeshow)
n_bins = 10
sorted_idx = np.argsort(p_test)
p_sorted, y_sorted = p_test[sorted_idx], y_test[sorted_idx]
bin_edges = np.array_split(np.arange(n_test), n_bins)
hl_stat = 0
cal_rows = []
for i, idx in enumerate(bin_edges):
    op, ep = y_sorted[idx].sum(), p_sorted[idx].sum()
    on, en = len(idx)-op, len(idx)-ep
    if ep>0 and en>0: hl_stat += (op-ep)**2/ep + (on-en)**2/en
    cal_rows.append({"Decile":i+1,"N":len(idx),
                     "Mean pred":round(p_sorted[idx].mean(),3),
                     "Actual":round(y_sorted[idx].mean(),3),
                     "Diff":round(y_sorted[idx].mean()-p_sorted[idx].mean(),3)})
hl_df   = pd.DataFrame(cal_rows)
hl_pval = 1 - chi2.cdf(hl_stat, df=n_bins-2)
print(f"\n── CALIBRATION")
print(f"  H-L χ²={hl_stat:.2f}  p={hl_pval:.4f}  "
      f"{'well-calibrated ✅' if hl_pval>0.05 else 'miscalibrated ⚠️'}")
print(hl_df.to_string(index=False))

# 4. Leakage
leakage_flags = []
for feat in avail:
    col = X_test[feat].fillna(X_test[feat].median())
    if col.std()==0: continue
    try:
        fa = roc_auc_score(y_test, col)
        fa = max(fa, 1-fa)
        if fa > 0.62: leakage_flags.append((feat,fa))
    except: pass
print(f"\n── LEAKAGE CHECK")
if leakage_flags:
    for feat, fa in sorted(leakage_flags, key=lambda x: -x[1]):
        print(f"  ⚠️  {feat:<40} AUC={fa:.4f}")
else:
    print(f"  ✅ No individual feature exceeds AUC 0.62")

# 5. Temporal stability
df_test = df.iloc[split_idx:].copy()
df_test["p_hr"] = p_test
df_test["year"] = pd.to_datetime(df_test["game_date"]).dt.year
print(f"\n── TEMPORAL STABILITY")
yr_aucs = []
for yr, grp in df_test.groupby("year"):
    if len(grp)<200: continue
    ya = roc_auc_score(grp["hr"], grp["p_hr"])
    yr_aucs.append(ya)
    print(f"  {yr}: N={len(grp):,}  AUC={ya:.4f}  HR%={grp['hr'].mean():.3f}")
if len(yr_aucs)>1:
    print(f"  AUC std: {np.std(yr_aucs):.4f}  "
          f"{'stable ✅' if np.std(yr_aucs)<0.02 else 'variable ⚠️'}")

# 6. Overfit
X_train = X.iloc[:split_idx].copy()
y_train = y[:split_idx]
p_train = booster_eval.predict(xgb.DMatrix(X_train, feature_names=avail))
overfit_gap = roc_auc_score(y_train, p_train) - auc
print(f"\n── OVERFIT")
print(f"  Train AUC: {roc_auc_score(y_train, p_train):.4f} | Test AUC: {auc:.4f} | "
      f"Gap: {overfit_gap:.4f}  {'acceptable ✅' if overfit_gap<0.03 else 'overfit ⚠️'}")

# 7. HR bucket monotonicity
print(f"\n── HR RATES BY PROBABILITY BUCKET")
for lo, hi in [(0.00,0.08),(0.08,0.12),(0.12,0.16),(0.16,0.20),(0.20,0.30),(0.30,1.0)]:
    mask = (p_test>=lo) & (p_test<hi)
    if mask.sum()<10: continue
    hr = y_test[mask].mean()
    print(f"  p=[{lo:.2f},{hi:.2f})  N={mask.sum():>6,}  HR%={hr:.3f}  "
          f"{'✅' if hr>lo else '⚠️'}")

# 8. SHAP top features (quick — full analysis in Section 14b)
print(f"\n── SHAP IMPORTANCE (top 10)")
explainer   = shap.TreeExplainer(booster_eval)
shap_sample = X_test.sample(min(5000,len(X_test)), random_state=42)
shap_values = explainer.shap_values(shap_sample)
shap_imp    = pd.Series(np.abs(shap_values).mean(axis=0),
                        index=avail).sort_values(ascending=False)
for feat, val in shap_imp.head(10).items():
    print(f"  {feat:<38} {val:.5f}")

# Scorecard
checks = [
    ("AUC > 0.60",             auc>0.60,              f"{auc:.4f}"),
    ("AUC CI excludes 0.5",    ci_lo>0.5,             f"[{ci_lo:.4f},{ci_hi:.4f}]"),
    ("AUC p < 0.001",          p_auc<0.001,           f"p={p_auc:.2e}"),
    ("PR-AUC skill > 0",       prauc_skill>0,         f"{prauc_skill:+.4f}"),
    ("Brier skill > 0",        brier_skill>0,         f"{brier_skill:+.4f}"),
    ("Calibration OK (H-L)",   hl_pval>0.05,          f"p={hl_pval:.4f}"),
    ("No leakage",             len(leakage_flags)==0, f"{len(leakage_flags)} flags"),
    ("Overfit gap < 0.03",     overfit_gap<0.03,      f"{overfit_gap:.4f}"),
    ("Pred std healthy",       p_test.std()>0.02,     f"{p_test.std():.4f}"),
    ("Platoon top feature",    shap_imp.index[0] in ["batter_hr_rate_vs_hand","batter_hr_rate_L50","batter_hr_zone_rate_L20"],
                                                      f"{shap_imp.index[0]}"),
]
print(f"\n{'='*70}")
print("SUMMARY SCORECARD")
print(f"{'='*70}")
passed = sum(1 for _,ok,_ in checks if ok)
for label, ok, detail in checks:
    print(f"  {'✅' if ok else '⚠️ '}  {label:<38}  {detail}")
print(f"\n  Score: {passed}/{len(checks)} checks passed")
if passed>=len(checks)-1: print("  Model passes rigor checks.")
else: print("  Review flagged items before increasing stakes.")
print(f"{'='*70}")

  HR PRO v6 — MATHEMATICAL RIGOR ASSESSMENT
✅ OOS model loaded

── CORE METRICS
  Test rows:    41,929 | Base HR rate: 0.111
  AUC:          0.6333
  PR-AUC:       0.1731  (baseline=0.111)  skill=+0.0694
  Brier skill:  +0.0226
  Cal gap:      0.0009

── AUC CONFIDENCE INTERVAL
  95% CI: [0.6244, 0.6423]  p=0.00e+00  significant ✅

── CALIBRATION
  H-L χ²=9.97  p=0.2670  well-calibrated ✅
 Decile    N  Mean pred  Actual   Diff
      1 4193      0.044   0.044  0.000
      2 4193      0.064   0.060 -0.005
      3 4193      0.079   0.076 -0.003
      4 4193      0.092   0.088 -0.003
      5 4193      0.103   0.097 -0.006
      6 4193      0.114   0.112 -0.003
      7 4193      0.127   0.128  0.001
      8 4193      0.141   0.151  0.010
      9 4193      0.159   0.164  0.005
     10 4192      0.199   0.194 -0.005

── LEAKAGE CHECK
  ✅ No individual feature exceeds AUC 0.62

── TEMPORAL STABILITY
  2025: N=41,929  AUC=0.6333  HR%=0.111

── OVERFIT
  Train AUC: 0.6476 | Test AUC: 0.6333 | Ga

## Section 14b: SHAP Analysis (NEW v6)
Run after retrain. Use OOS model. Identifies prune candidates before pushing to production.

**Usage:** `shap_df = run_shap_analysis(cfg)`

In [27]:
def run_shap_analysis(cfg, feat_df=None, top_n=30, sample_n=5000):
    """
    Full SHAP analysis on OOS test set.
    Returns shap_df sorted by mean |SHAP| descending.
    Flags features with < 2% of top feature's SHAP as prune candidates.
    """
    try:
        import shap
    except ImportError:
        print("⚠️  pip install shap"); return

    if feat_df is None: feat_df = features
    m = model.get("hr")
    if m is None: print("⚠️  No model loaded"); return

    if model.get("meta",{}).get("full_retrain"):
        print("⚠️  Full retrain model loaded — SHAP values will be in-sample")
        oos_path = cfg["model_xgb"].replace(".json","_oos.json")
        if Path(oos_path).exists():
            m_oos = xgb.Booster()
            m_oos.load_model(oos_path)
            print(f"   Auto-loading OOS model from {oos_path}")
            m = m_oos
        else:
            print("   No OOS model found — proceeding with full retrain (in-sample SHAP)")

    avail = [f for f in HR_FEATURES if f in feat_df.columns]
    df    = feat_df.dropna(subset=["hr"]+avail).sort_values("game_date")
    split = model["meta"].get("split_date")
    test  = df[df["game_date"].astype(str) >= split] if split else df

    if len(test) > sample_n:
        test = test.sample(sample_n, random_state=42)

    X_s  = test[avail].astype(float)
    expl = shap.TreeExplainer(m)
    vals = expl.shap_values(X_s)

    shap_df = pd.DataFrame({
        "feature":   avail,
        "mean_shap": np.abs(vals).mean(axis=0),
    }).sort_values("mean_shap", ascending=False).reset_index(drop=True)

    top_shap = shap_df["mean_shap"].iloc[0]
    threshold = top_shap * 0.02

    print(f"\n── SHAP FEATURE IMPORTANCE (OOS test, n={len(test):,})")
    print(f"   {'Feature':<35} {'Mean |SHAP|':>12}  {'Rank':>4}  {'Bar'}")
    print(f"   {'─'*35} {'─'*12}  {'─'*4}  {'─'*20}")
    for i, row in shap_df.head(top_n).iterrows():
        bar = "█" * int(row["mean_shap"] / top_shap * 20)
        flag = "  ← prune?" if row["mean_shap"] < threshold else ""
        print(f"   {row['feature']:<35} {row['mean_shap']:>12.4f}  {i+1:>4}  {bar}{flag}")

    # Prune candidates
    low = shap_df[shap_df["mean_shap"] < threshold]
    if not low.empty:
        print(f"\n  ⚠️  Prune candidates (< 2% of top feature SHAP):")
        for _, row in low.iterrows():
            print(f"     drop '{row['feature']}',  # mean SHAP={row['mean_shap']:.5f}")
        print(f"\n  To prune, remove from HR_FEATURES in Section 0 and rebuild.")
    else:
        print(f"\n  ✅ No obvious prune candidates at 2% threshold.")

    # Calibration check
    dm    = xgb.DMatrix(X_s, feature_names=avail)
    preds = m.predict(dm)
    y     = test["hr"].values
    print(f"\n  Mean predicted: {preds.mean():.4f} | Actual: {y.mean():.4f} | "
          f"Gap: {abs(preds.mean()-y.mean()):.4f}")

    # Feature group summary
    groups = {
        "Batter contact":     [f for f in avail if "batter" in f and any(x in f for x in ["hr","barrel","hard","ev","la","sweet","zone","fb","xwoba","launch"])],
        "Batter swing":       [f for f in avail if any(x in f for x in ["whiff","chase","contact_pct","swing"])],
        "Platoon":            [f for f in avail if "vs_hand" in f or "vs_lhp" in f or "vs_rhp" in f],
        "Pitcher":            [f for f in avail if f.startswith("pitcher_") or f=="starter_avg_ip_L5"],
        "Park/Environment":   [f for f in avail if any(x in f for x in ["park","dist","altitude","air","dome"])],
        "Weather":            [f for f in avail if any(x in f for x in ["temp","wind","cold","hot"])],
        "Context":            [f for f in avail if any(x in f for x in ["order","win_pct","moneyline"])],
    }
    print(f"\n── FEATURE GROUP SHAP SUMMARY")
    for grp, feats in groups.items():
        grp_shap = shap_df[shap_df["feature"].isin(feats)]["mean_shap"].sum()
        pct      = grp_shap / shap_df["mean_shap"].sum() * 100
        print(f"  {grp:<22} {grp_shap:.4f}  ({pct:.1f}% of total)")

    return shap_df

# Run analysis
shap_df = run_shap_analysis(cfg)

⚠️  Full retrain model loaded — SHAP values will be in-sample
   Auto-loading OOS model from C:\Users\lmayn\Downloads\HR_Pro\models\xgb_hr_v6_oos.json

── SHAP FEATURE IMPORTANCE (OOS test, n=5,000)
   Feature                              Mean |SHAP|  Rank  Bar
   ─────────────────────────────────── ────────────  ────  ────────────────────
   batter_barrel_rate_L50                    0.1263     1  ████████████████████
   batter_hr_rate_vs_hand                    0.1114     2  █████████████████
   batter_max_ev_L20                         0.0917     3  ██████████████
   batter_hr_rate_L50                        0.0644     4  ██████████
   batter_hr_rate_season                     0.0602     5  █████████
   hr_park_factor                            0.0480     6  ███████
   ewma_batting_order                        0.0443     7  ███████
   temperature_f                             0.0428     8  ██████
   pitcher_fb_rate_L50                       0.0392     9  ██████
   batter_zone_contact

## Build Order

### First-time setup
```python
statcast_historical(cfg)             # 3-5 hours overnight
weather_historical(cfg)              # ~45 min
build_game_features_native(cfg)      # ~10 min
build_batting_order_map(cfg)
build_player_id_map(cfg)
build_savant_hr_park_factors()       # seasonal, Selenium required

build_player_game(cfg, incremental=False)
build_batter_rolling(cfg, lookback_days=9999)   # v6 full rebuild
build_platoon_features(cfg)
build_pitcher_hr_features(cfg, lookback_days=9999)  # v6 full rebuild
build_features(cfg)

train_model(cfg)
walk_forward_cv(cfg)                 # honest AUC
run_shap_analysis(cfg)               # prune before production
full_retrain(cfg)
```

### v6 rebuild from v5 CSVs
```python
# Required — new columns not in v5 cached files
build_batter_rolling(cfg, lookback_days=9999)
build_pitcher_hr_features(cfg, lookback_days=9999)
build_features(cfg)
train_model(cfg)
run_shap_analysis(cfg)
full_retrain(cfg)
```

### v6 TODO (deferred)
| Issue | Impact | Fix |
|---|---|---|
| Flat 7% vig strip | Medium | Calibrate after 200+ settled rows |
| Implied run total (over/under) | Low-Med | Add to DK scraper |
| 2021 hc_x 67% | Low | `statcast_repull_recent(cfg, seasons=(2021,))` |
| Bat tracking features | Low | Mid-2023 onward only — too thin yet |

In [28]:
build_features(cfg)
train_model(cfg)
walk_forward_cv(cfg)
run_shap_analysis(cfg)
full_retrain(cfg)

Building feature table from 249,251 player-game rows...
  After batter join: 249,251 | matched: 236,781
  After pitcher join: 242,239 matched (97.2%)
  After weather join: 182,752 matched (73.3%)
  After game features join: 234,387 matched
  After order join: 249,251 matched
  After platoon join: 100.0% coverage
  ✅ 249,251 rows | Features: 22/22 | HR rate: 0.107
Training on 22 features
  Train: 167,715 through 2025-04-13
  Test:  41,929 from 2025-04-13
  HR rates: train=0.112 test=0.111
[0]	train-logloss:0.35072	train-auc:0.60976	test-logloss:0.34892	test-auc:0.61250
[100]	train-logloss:0.34017	train-auc:0.63400	test-logloss:0.33895	test-auc:0.63135
[200]	train-logloss:0.33863	train-auc:0.64112	test-logloss:0.33832	test-auc:0.63330
[288]	train-logloss:0.33782	train-auc:0.64520	test-logloss:0.33826	test-auc:0.63322
  OOS model saved → C:\Users\lmayn\Downloads\HR_Pro\models\xgb_hr_v6_oos.json

OOS RESULTS (test from 2025-04-13)
  AUC:        0.6332
  Brier:      0.0968
  Log loss:   0.3